# Phase 2 — Hybrid RAG

Requires the provided Phase-1 Python export and saved Phase-1 outputs. BM25, RRF, context packing, citation parsing and gold-mask testing are implemented in this notebook.

```text
/kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/
├── nlp-phase1.py
├── course_pack_v1/course_pack_v1/{corpus.jsonl,questions_dev.jsonl,labels_dev.jsonl,questions_hidden.jsonl,score.py}
├── embeddings/{document_embeddings_4B.npz,document_embeddings_4B.meta.json}
└── NLP_phase1_output/dev/dense/{predictions.jsonl,run_config.yaml}
```

Use Kaggle T4 x2 and Internet. Set the Git commit in CONFIG. Run all cells in order. `report.pdf` is the official three-page report; `report_detailed.pdf` provides extended analysis. Both reports and their HTML sources are generated from the current run.


In [1]:
# =============================================================================
# §0-1  Environment setup: install dependencies, print versions, check GPU
# =============================================================================
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

_pip("-U", "transformers>=4.51.0", "accelerate", "pyyaml",
     "weasyprint>=62,<71", "pypdf", "rank_bm25")
print("dependencies ready")

import os, re, gc, json, math, time, random, string, zipfile, shutil
from pathlib import Path
from collections import Counter, defaultdict
from datetime import date

import numpy as np

import torch
import torch.nn.functional as F

# Enable PyTorch's expandable-segment allocator to reduce VRAM fragmentation,
# and turn on memory-efficient / flash scaled-dot-product attention when available.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
if hasattr(torch.backends.cuda, "enable_sdp_backends"):
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_flash_sdp(True)

from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import transformers

try:
    import accelerate
    ACCELERATE_VERSION = accelerate.__version__
except ImportError:
    ACCELERATE_VERSION = "NOT INSTALLED (required for device_map='auto')"

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("NumPy       :", np.__version__)
print("transformers:", transformers.__version__)
print("accelerate  :", ACCELERATE_VERSION)

assert torch.cuda.is_available(), (
    "A GPU is required. In Kaggle set Accelerator = GPU T4 x2."
)
print("Number of GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | {props.total_memory / 1024**3:.1f} GiB")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.empty_cache()
gc.collect()


dependencies ready
Python      : 3.13.16
PyTorch     : 2.11.0+cu130
NumPy       : 2.1.3
transformers: 5.23.0
accelerate  : 1.16.0
Number of GPUs: 2
  GPU 0: Tesla T4 | 14.6 GiB
  GPU 1: Tesla T4 | 14.6 GiB


169

In [2]:
# =============================================================================
# §0-2  Central CONFIG — all paths, model choices and hyper-parameters live here
# =============================================================================
# Each run writes a `run_config.yaml` snapshot of this dict alongside predictions
# so every result is reproducible.

STD_ID       = "404722145"
STUDENT_NAME = "محمدحسین کریمی"
GITHUB_URL   = "https://github.com/mohammad-hossein-k-gh/Advanced-NLP_Phase1"
GITHUB_COMMIT = "5d7d445"  # Provided commit; verify it corresponds to the submitted Phase-2 code.
REPORT_DATE = "۱۴۰۵/۰۸/۰۱"  # User-provided report date, not automatically the execution date.

# -----------------------------------------------------------------------------
# Pinned Kaggle dataset path (must match the path Kaggle mounts your dataset at):
#   /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/
#       ├── course_pack_v1/course_pack_v1/{corpus.jsonl, questions_dev.jsonl, ...}
#       ├── NLP_phase1_output/dev/dense/{predictions.jsonl, run_config.yaml}
#       ├── embeddings/{document_embeddings_4B.npz, document_embeddings_4B.meta.json}
#       └── nlp-phase1.py
# -----------------------------------------------------------------------------
KAGGLE_DATASET_ROOT = "/kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset"
PACK_SUBDIR         = "course_pack_v1"

CONFIG = {
    # ---------- paths ----------
    "dataset_root": KAGGLE_DATASET_ROOT,
    # Alternate roots kept for compatibility with older Kaggle dataset slug layouts.
    # The pinned root above is always checked first.
    "dataset_root_alternates": [
        "/kaggle/input/nlp-phase1-dataset",
        "/kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset",
    ],
    "pack_subdir": PACK_SUBDIR,
    "phase1_outputs_dir": None,          # None = auto-detect Phase-1 artifacts
    "pack_dir": None,                    # override manually if needed; None = dataset_root/course_pack_v1
    "output_dir": "/kaggle/working/NLP_phase2",

    # ---------- models (kept identical to Phase 1) ----------
    "embedding_model": "Qwen/Qwen3-Embedding-4B",
    "llm_model": "Qwen/Qwen3-8B",
    # `max_memory` is passed to accelerate's `device_map="auto"`.
    # Keys MUST be integers (newer accelerate rejects string keys like "0"/"1").
    # Per-GPU cap is intentionally below total VRAM so headroom remains for
    # KV-cache activations during generation (the 8B generator needs ~1-2 GiB
    # of free VRAM per GPU for activations and KV-cache).
    "gpu_memory": {0: "12GiB", 1: "12GiB", "cpu": "40GiB"},

    # ---------- embedding ----------
    # Phase-1 document embedding call uses batch_size=4 and max_length=1024.
    # Capacity and truncation depend on actual data and hardware.
    "embed_batch": 4,
    "embed_max_len": 1024,
    # Phase-2 sentence-level experiment; tune batch size for available VRAM.
    "sent_batch": 16,
    "sent_max_len": 512,

    # ---------- retrieval ----------
    # fusion_depth=100: each retriever returns 100 candidates before RRF.
    # Dense is a single matmul (cheap); BM25 over 3882 docs is also cheap.
    # 100 is a configurable candidate depth, not a measured optimum.
    "fusion_depth": 100,
    "k_output": 10,                # number of IDs written to `retrieved` (score.py KS=(2,5,10))
    "k_context": 10,               # cap on passages considered during packing
    "rrf_k": 60,                   # standard RRF constant (Cormack et al.; handout §9.1)
    "bm25_k1": 1.5,                # standard BM25 defaults (handout §9.1)
    "bm25_b": 0.75,
    "bm25_remove_stopwords": True,  # handout: remove stopwords before indexing

    # ---------- generation ----------
    # Start at batch 2; retry with smaller batches on CUDA out-of-memory.
    # Memory requirements and numerical effects depend on the runtime.
    "llm_batch": 2,
    "max_new_tokens": 96,          # Phase-2 generation limit; the Phase-1 function default was 128
    "max_prompt_tokens": 4096,     # Phase 1 truncation limit
    # Set a conservative context budget; measure its effect on this dataset.
    "max_context_tokens": 1800,
    "temperature": 0.0,            # greedy decoding (do_sample=False)

    # ---------- execution ----------
    "dev_limit": None,             # set to a small number (e.g. 20) for quick smoke runs only
    "run_dev": True,
    "run_sentence": True,
    "run_goldmask": True,
    "hidden_systems": ["hybrid"],
}

# Guard against accidentally using string GPU keys (older accelerate raises a
# cryptic "Device 0 is not recognized" error).
assert all(isinstance(k, int) or k in ("cpu", "disk", "mps")
           for k in CONFIG["gpu_memory"]), (
    "Keys in gpu_memory must be int (e.g. 0, 1); string keys like '0' are rejected by accelerate."
)

OUTPUT_DIR = Path(CONFIG["output_dir"])
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
RUNS_DIR = OUTPUT_DIR / "runs"
PROMPT_DIR = OUTPUT_DIR / "prompts"
for _d in (OUTPUT_DIR, ARTIFACT_DIR, RUNS_DIR, PROMPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print("Dataset root (pinned):", CONFIG["dataset_root"])
print("Pack subdir          :", CONFIG["pack_subdir"])
print("Output dir           :", OUTPUT_DIR)
print("\nCONFIG:")
for k, v in CONFIG.items():
    if k not in ("gpu_memory", "dataset_root_alternates"):
        print(f"  {k:22s} = {v}")


Dataset root (pinned): /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset
Pack subdir          : course_pack_v1
Output dir           : /kaggle/working/NLP_phase2

CONFIG:
  dataset_root           = /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset
  pack_subdir            = course_pack_v1
  phase1_outputs_dir     = None
  pack_dir               = None
  output_dir             = /kaggle/working/NLP_phase2
  embedding_model        = Qwen/Qwen3-Embedding-4B
  llm_model              = Qwen/Qwen3-8B
  embed_batch            = 4
  embed_max_len          = 1024
  sent_batch             = 16
  sent_max_len           = 512
  fusion_depth           = 100
  k_output               = 10
  k_context              = 10
  rrf_k                  = 60
  bm25_k1                = 1.5
  bm25_b                 = 0.75
  bm25_remove_stopwords  = True
  llm_batch              = 2
  max_new_tokens         = 96
  max_prompt_tokens      = 4096
  max_context_tokens     = 1800
  temperature       

In [3]:
# =============================================================================
# §1  Resolve the dataset path and load course_pack_v1
# =============================================================================
# Phase-2-only helpers. These functions are not defined in Phase 1.
def extract_short_answer(raw_text):
    text = re.sub('<think>.*?</think>', '', raw_text, flags=re.DOTALL)
    text = re.sub('\\s+', ' ', text).strip()
    matches = list(re.finditer('answer\\s*[:：]\\s*', text, flags=re.IGNORECASE))
    if matches:
        text = text[matches[-1].end():].strip()
    text = re.sub(r'\[\s*p\d{4,6}\s*\]', '', text, flags=re.IGNORECASE)
    text = re.sub('^(the answer is|therefore|so|thus|final answer)[,:\\s]*', '', text, flags=re.IGNORECASE).strip()
    text = text.strip('"\'“”«»')
    m = re.fullmatch('(p\\d{4,6})', text, flags=re.IGNORECASE)
    if m:
        return m.group(1).lower()
    if len(text.split()) > 10:
        text = re.split('(?<=[.!?])\\s+', text)[0]
    return re.sub('[\\s.,;:!?]+$', '', text).strip()


def generate_from_prompts(prompts, tokenizer, model, max_new_tokens=96, max_length=4096, pad_token_id=None):
    import torch
    if pad_token_id is None:
        pad_token_id = getattr(tokenizer, 'pad_token_id', None)
        if pad_token_id is None:
            pad_token_id = getattr(tokenizer, 'eos_token_id', None)
    enc = tokenizer(prompts, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    device = model.get_input_embeddings().weight.device
    enc = {k: v.to(device) for k, v in enc.items()}
    prompt_len = enc['input_ids'].shape[1]
    with torch.inference_mode():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True, pad_token_id=pad_token_id, no_repeat_ngram_size=8, repetition_penalty=1.05)
    new_tokens = out[:, prompt_len:].detach().cpu()
    del out, enc
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)


def invoke_course_scorer(pred_path, score_path, pack_dir, split='dev', compare=None, partial=False, validate_only=False):
    import subprocess
    import sys
    cmd = [sys.executable, str(score_path), str(pred_path), '--pack', str(pack_dir), '--split', split]
    if compare:
        cmd += ['--compare', str(compare)]
    if partial:
        cmd += ['--partial']
    if validate_only:
        cmd += ['--validate-only']
    res = subprocess.run(cmd, text=True, capture_output=True)
    if res.returncode != 0:
        print(res.stdout)
        print(res.stderr)
        raise RuntimeError(f"score.py failed: {' '.join(cmd)}")
    return res.stdout


def parse_score_output(text):
    import re as _re
    overall, by_type, paired, type_n = ({}, {}, {}, {})
    overall_n, section, cur = (0, None, None)
    metric_re = _re.compile('^\\s+([A-Za-z0-9_@]+)\\s+([+-]?\\d+(?:\\.\\d+)?)\\s+\\[\\s*([+-]?\\d+(?:\\.\\d+)?),\\s*([+-]?\\d+(?:\\.\\d+)?)\\s*\\]')
    for line in text.splitlines():
        m = _re.match('^overall\\s*\\(n=(\\d+)\\)', line)
        if m:
            overall_n, section, cur = (int(m.group(1)), 'overall', None)
            continue
        m = _re.match('^type:\\s*(\\S+)\\s*\\(n=(\\d+)\\)', line)
        if m:
            section, cur = ('type', m.group(1))
            by_type[cur] = {}
            type_n[cur] = int(m.group(2))
            continue
        if line.startswith('paired difference'):
            section, cur = ('paired', None)
            continue
        m = metric_re.match(line)
        if m:
            key, val = (m.group(1), float(m.group(2)))
            lo, hi = (float(m.group(3)), float(m.group(4)))
            entry = {'value': val, 'ci': (lo, hi)}
            if section == 'overall':
                overall[key] = entry
            elif section == 'type':
                by_type[cur][key] = entry
            elif section == 'paired':
                paired[key] = entry
    return {'overall': overall, 'by_type': by_type, 'paired': paired, 'type_n': type_n, 'overall_n': overall_n}

# Load used functions from Phase 1 without running its top-level experiments.
# A normal import would execute model loading and all notebook-export cells.
import ast
import torch.nn.functional as F

_configured_roots = [Path(CONFIG["dataset_root"])] + [Path(x) for x in CONFIG.get("dataset_root_alternates", [])]
_source_roots = [root for root in _configured_roots if (root / "nlp-phase1.py").is_file()]
if not _source_roots:
    raise FileNotFoundError("nlp-phase1.py not found in the configured dataset root or alternates.")
PHASE1_SOURCE_PATH = _source_roots[0] / "nlp-phase1.py"
phase1_source = PHASE1_SOURCE_PATH.read_text(encoding="utf-8")
phase1_tree = ast.parse(phase1_source, filename=str(PHASE1_SOURCE_PATH))
for source_name, config_name in (("EMBEDDING_MODEL_NAME", "embedding_model"), ("LLM_MODEL_NAME", "llm_model")):
    values = []
    for node in phase1_tree.body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == source_name for t in node.targets):
            try:
                values.append(ast.literal_eval(node.value))
            except (ValueError, TypeError):
                pass
    if not values or any(v != CONFIG[config_name] for v in values):
        raise ValueError(f"Phase-1 {source_name} does not match CONFIG[{config_name!r}].")

# Names stay identical to the supplied .py; the functions run in this notebook's globals.
phase1_function_names = {"read_jsonl", "make_embeddings", "dense_search", "save_predictions"}
phase1_nodes = [n for n in phase1_tree.body if isinstance(n, ast.FunctionDef) and n.name in phase1_function_names]
missing = phase1_function_names - {n.name for n in phase1_nodes}
if missing:
    raise RuntimeError(f"Phase-1 export is missing definitions: {sorted(missing)}")
exec(compile(ast.Module(body=phase1_nodes, type_ignores=[]), str(PHASE1_SOURCE_PATH), "exec"), globals())
# Preserve the original dense function before defining the Phase-2 vector interface.
phase1_dense_search = dense_search
required_fields_nodes = [n for n in phase1_tree.body if isinstance(n, ast.Assign)
                         and any(isinstance(t, ast.Name) and t.id == "REQUIRED_FIELDS" for t in n.targets)]
if not required_fields_nodes:
    raise RuntimeError("Phase-1 export is missing REQUIRED_FIELDS.")
REQUIRED_FIELDS = ast.literal_eval(required_fields_nodes[0].value)
query_assignments = [n for n in phase1_tree.body if isinstance(n, ast.Assign)
                     and any(isinstance(t, ast.Name) and t.id == "query_texts" for t in n.targets)
                     and isinstance(n.value, ast.ListComp)]
if not query_assignments or not isinstance(query_assignments[0].value.elt, ast.BinOp):
    raise RuntimeError("Could not locate the Phase-1 query instruction.")
try:
    QUERY_INSTRUCT = ast.literal_eval(query_assignments[0].value.elt.left)
except (ValueError, TypeError) as exc:
    raise RuntimeError("Could not safely read the Phase-1 query instruction.") from exc


def query_texts_for(items):
    return [QUERY_INSTRUCT + item["question"] for item in items]


def dense_search_from_scores(scores, doc_ids, k=10, exclude_ids=frozenset()):
    # Phase-1 dense_search reads these exact global names.
    global question_text_index, similarities, saved_doc_ids, doc_id_to_index
    question_text_index = {"__phase2_query__": 0}
    similarities = np.asarray(scores)[None, :]
    saved_doc_ids = [str(x) for x in doc_ids]
    doc_id_to_index = {d: i for i, d in enumerate(saved_doc_ids)}
    return phase1_dense_search("__phase2_query__", k=k, exclude_ids=exclude_ids)

print(f"Phase-1 functions loaded from {PHASE1_SOURCE_PATH}: {sorted(phase1_function_names)}")

# Required files inside the course pack.
PACK_FILES = ["corpus.jsonl", "questions_dev.jsonl",
              "labels_dev.jsonl", "questions_hidden.jsonl"]
OPTIONAL_FILES = ["score.py", "README.txt", "kg_subset_passages.txt",
                  "LICENSE_AND_ATTRIBUTION.txt"]


def _pack_is_complete(d: Path) -> bool:
    """Return True if directory *d* contains all required pack files."""
    return all((d / f).is_file() for f in PACK_FILES)


def resolve_dataset_root():
    """Find the complete course pack under the Phase-1 source root."""
    for root in [PHASE1_SOURCE_PATH.parent]:
        if not root.exists():
            continue
        for sub in (root / CONFIG["pack_subdir"] / CONFIG["pack_subdir"], root / CONFIG["pack_subdir"], root):
            if sub.is_dir() and _pack_is_complete(sub):
                print(f"dataset root resolved: {root}")
                return root

    raise FileNotFoundError(
        f"The course pack was not found under the same dataset root as {PHASE1_SOURCE_PATH}. "
        "Check CONFIG['dataset_root'] and the nested course_pack_v1 folders."
    )

DATASET_ROOT = resolve_dataset_root()

# List dataset contents (mirrors Phase 1's /kaggle/input walk but scoped to our dataset).
print("\nDataset contents:")
_shown = 0
for _root, _dirs, _files in os.walk(DATASET_ROOT):
    for _f in sorted(_files):
        rp = Path(_root) / _f
        try:
            rel = rp.relative_to(DATASET_ROOT)
        except ValueError:
            rel = rp
        print(" ", rel)
        _shown += 1
        if _shown >= 80:
            print("  ... (truncated)")
            break
    if _shown >= 80:
        break


def find_pack_dir():
    """Return the directory that contains the JSONL pack files."""
    override = CONFIG.get("pack_dir")
    if override:
        d = Path(override)
        if not _pack_is_complete(d):
            raise FileNotFoundError(f"Configured pack_dir {d} does not contain all required files.")
        return d

    primary = DATASET_ROOT / CONFIG["pack_subdir"] / CONFIG["pack_subdir"]
    if _pack_is_complete(primary):
        return primary
    primary = DATASET_ROOT / CONFIG["pack_subdir"]
    if _pack_is_complete(primary):
        return primary

    # Files are loose at the dataset root (non-standard layout).
    if _pack_is_complete(DATASET_ROOT):
        print(f"WARNING: pack files are at dataset root rather than {CONFIG['pack_subdir']}/")
        return DATASET_ROOT

    missing = [f for f in PACK_FILES if not (DATASET_ROOT / f).is_file()
               and not (primary / f).is_file()]
    raise FileNotFoundError(
        f"Course pack not found under {primary}. Missing files: {missing}"
    )


PACK_DIR = find_pack_dir()
print("\nPack dir:", PACK_DIR)
for f in PACK_FILES + OPTIONAL_FILES:
    print(f"  {'OK ' if (PACK_DIR / f).exists() else '--'} {f}")


def resolve_score_script():
    """Locate score.py in the pack, dataset root, or anywhere under /kaggle/input."""
    candidates = [PACK_DIR / "score.py", DATASET_ROOT / "score.py"]
    search_roots = [Path("/kaggle/input")] if Path("/kaggle/input").exists() else []
    search_roots += [Path.cwd(), Path("/home/user")]
    for sr in search_roots:
        try:
            candidates += list(sr.rglob("score.py"))
        except OSError:
            pass
    for c in candidates:
        if c.is_file():
            return c
    raise FileNotFoundError(
        "score.py not found. It should be inside the course pack or at the dataset root."
    )


SCORE_PATH = resolve_score_script()
print("\nscore.py:", SCORE_PATH)


corpus_path         = PACK_DIR / "corpus.jsonl"
questions_dev_path  = PACK_DIR / "questions_dev.jsonl"
labels_dev_path     = PACK_DIR / "labels_dev.jsonl"
questions_hidden_path = PACK_DIR / "questions_hidden.jsonl"

corpus            = read_jsonl(corpus_path)
questions_dev     = read_jsonl(questions_dev_path)
labels_dev        = read_jsonl(labels_dev_path)
questions_hidden  = read_jsonl(questions_hidden_path)

print("\nLoaded:")
print("  corpus          :", len(corpus), "passages  -", corpus_path)
print("  questions_dev   :", len(questions_dev), "questions -", questions_dev_path)
print("  labels_dev      :", len(labels_dev), "labels    -", labels_dev_path)
print("  questions_hidden:", len(questions_hidden), "questions -", questions_hidden_path)

# Reuse the original Phase-1 document formatting loop without executing the full script.
phase1_document_loops = [node for node in phase1_tree.body if isinstance(node, ast.For)
                         and isinstance(node.target, ast.Name) and node.target.id == "item"
                         and isinstance(node.iter, ast.Name) and node.iter.id == "corpus"
                         and any(isinstance(child, ast.Call) and isinstance(child.func, ast.Attribute)
                                 and isinstance(child.func.value, ast.Name)
                                 and child.func.value.id == "doc_texts" and child.func.attr == "append"
                                 for child in ast.walk(node))]
if len(phase1_document_loops) != 1:
    raise RuntimeError("Could not identify the unique Phase-1 document formatting loop.")
doc_ids, doc_texts = [], []
exec(compile(ast.Module(body=phase1_document_loops, type_ignores=[]), str(PHASE1_SOURCE_PATH), "exec"), globals())
DOC_IDS = [str(d) for d in doc_ids]
PASSAGE_TEXTS = doc_texts
DOC_IDX = {d: i for i, d in enumerate(DOC_IDS)}
assert len(DOC_IDS) == len(DOC_IDX), "duplicate doc_ids in corpus"
DOC_TITLE, DOC_TEXT, DOC_SENTS = {}, {}, {}
for item in corpus:
    doc_id = str(item["doc_id"])
    if doc_id not in DOC_IDX:
        continue
    DOC_TITLE[doc_id] = str(item.get("title", "")).strip()
    DOC_TEXT[doc_id] = str(item.get("text", "")).strip() or " ".join(item.get("sentences", [])).strip()
    DOC_SENTS[doc_id] = [str(x).strip() for x in (item.get("sentences") or []) if str(x).strip()]
print(f"Document formatting reused directly from Phase 1 ({len(DOC_IDS)} docs)")

LABELS_BY_QID = {str(l["qid"]): l for l in labels_dev}
QDEV_BY_QID = {str(q["qid"]): q for q in questions_dev}
assert set(LABELS_BY_QID) == set(QDEV_BY_QID), "qid mismatch between dev questions and labels"
assert set(str(q["qid"]) for q in questions_dev).isdisjoint(
    set(str(q["qid"]) for q in questions_hidden)), "dev and hidden qids overlap"

# Aggregate statistics.
from collections import Counter as _Counter
_type_counts = _Counter(str(q.get("type", "?")) for q in questions_dev)
_gold_counts = [len(l.get("gold_doc_ids", [])) for l in labels_dev]
print("\nDev question types:", dict(_type_counts))
print(f"Gold docs per question: mean={np.mean(_gold_counts):.2f} "
      f"min={min(_gold_counts)} max={max(_gold_counts)}")
_q0 = questions_dev[0]
print("\nSample dev question:")
print("  qid    :", _q0["qid"])
print("  type   :", _q0.get("type", "?"))
print("  question:", _q0["question"])
_l0 = LABELS_BY_QID[str(_q0["qid"])]
print("  gold   :", _l0.get("gold_doc_ids"), "answer:", _l0.get("answer"))


Phase-1 functions loaded from /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/nlp-phase1.py: ['dense_search', 'make_embeddings', 'read_jsonl', 'save_predictions']
dataset root resolved: /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset

Dataset contents:
  nlp-phase1.py
  NLP_phase1_output/dev/closed_book/predictions.jsonl
  NLP_phase1_output/dev/closed_book/run_config.yaml
  NLP_phase1_output/dev/dense/predictions.jsonl
  NLP_phase1_output/dev/dense/run_config.yaml
  NLP_phase1_output/dev/oracle/predictions.jsonl
  NLP_phase1_output/dev/oracle/run_config.yaml
  NLP_phase1_output/hidden/closed_book/predictions.jsonl
  NLP_phase1_output/hidden/closed_book/run_config.yaml
  NLP_phase1_output/hidden/dense/predictions.jsonl
  NLP_phase1_output/hidden/dense/run_config.yaml
  NLP_phase1_output/selfcheck/predictions.jsonl
  course_pack_v1/course_pack_v1/LICENSE_AND_ATTRIBUTION.txt
  course_pack_v1/course_pack_v1/README.txt
  course_pack_v1/course_pack_v1/corpus.jsonl
  c

## §1.5 Phase-1 artifacts

Selected functions are loaded directly from the Python export without running its top-level experiments. Saved Phase-1 dense dev predictions and the document embedding matrix are mandatory.


In [4]:
# =============================================================================
# §1.5 Discover required Phase-1 dataset artifacts
# =============================================================================
WANTED_P1_FILES = [
    "document_embeddings_4B.npz",
    "NLP_phase1_output/dev/dense/predictions.jsonl",
    "NLP_phase1_output/dev/dense/run_config.yaml",
]

_roots = []
if CONFIG["phase1_outputs_dir"]:
    _roots.append(Path(CONFIG["phase1_outputs_dir"]))
_roots.append(DATASET_ROOT)

_roots = [r for r in _roots if r.exists()]


def _find_phase1_artifact(rel, roots):
    """Look for a Phase-1 artifact at <root>/rel, <root>/NLP_phase1/rel, or via rglob.

    Phase-2 outputs under OUTPUT_DIR are explicitly excluded so we don't
    accidentally use our own files as "Phase-1" artifacts.
    """
    rel_p = Path(rel)
    direct, rglob_hits = [], []
    for root in roots:
        for cand in (root / "embeddings" / rel_p, root / rel_p, root / "NLP_phase1" / rel_p):
            if cand.is_file() and OUTPUT_DIR not in cand.parents and "NLP_phase2" not in str(cand):
                direct.append(cand)
        try:
            for hit in root.rglob(rel_p.name):
                if not hit.is_file():
                    continue
                if OUTPUT_DIR in hit.parents or "NLP_phase2" in str(hit):
                    continue
                if not str(hit).endswith(rel):
                    continue
                rglob_hits.append(hit)
        except OSError:
            pass
    if direct:
        return sorted(direct, key=lambda p: ("phase1" not in str(p).lower(), str(p)))[0]
    if rglob_hits:
        return sorted(rglob_hits, key=lambda p: ("phase1" not in str(p).lower(), str(p)))[0]
    return None


PHASE1_ARTIFACTS = {rel: _find_phase1_artifact(rel, _roots) for rel in WANTED_P1_FILES}

print("\nPhase-1 artifacts:")
for rel, path in PHASE1_ARTIFACTS.items():
    status = f"OK  {path}" if path else "--  (not found)"
    print(f"  {status}  {rel}")

# The Phase-1 predictions and document matrix are mandatory.
for _required in ("NLP_phase1_output/dev/dense/predictions.jsonl", "NLP_phase1_output/dev/dense/run_config.yaml", "document_embeddings_4B.npz"):
    if PHASE1_ARTIFACTS[_required] is None:
        raise FileNotFoundError(f"Required Phase-1 artifact not found: {_required}.")



Phase-1 artifacts:
  OK  /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/embeddings/document_embeddings_4B.npz  document_embeddings_4B.npz
  OK  /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/NLP_phase1_output/dev/dense/predictions.jsonl  NLP_phase1_output/dev/dense/predictions.jsonl
  OK  /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/NLP_phase1_output/dev/dense/run_config.yaml  NLP_phase1_output/dev/dense/run_config.yaml


## §2 Passage, query and sentence embeddings

The document embedding NPZ and metadata in the dataset are mandatory; loading or validation failures stop the run. Query and sentence vectors are independently computed or reused from valid caches.


In [5]:
# =============================================================================
# §2  Passage, query and sentence embeddings
# =============================================================================

# Query prefix comes from Phase 1 (QUERY_INSTRUCT is exported by §1 from
# the supplied Phase-1 Python export).
print("query prefix:", repr(QUERY_INSTRUCT))


def resolve_max_memory():
    """Build an int-keyed max_memory dict containing only GPUs that are actually visible.

    Newer versions of accelerate reject string keys like "0"/"1" with
    "Device 0 is not recognized", and also reject keys for GPUs that do not
    exist (e.g. key 1 when only one GPU is visible).
    """
    n = torch.cuda.device_count()
    if n == 0:
        return None                       # let device_map="auto" decide
    template = CONFIG.get("gpu_memory", {})
    caps = {}
    for i in range(n):
        caps[i] = template.get(i, template.get(str(i), "13GiB"))
    caps["cpu"] = template.get("cpu", "40GiB")
    print("max_memory (int keys):", caps)
    return caps


def load_causal(cls, name, **kw):
    """Robust wrapper around `cls.from_pretrained`.

    Tries, in order:
        1. the supplied kwargs,
        2. with torch_dtype renamed to dtype (newer transformers),
        3. additionally with max_memory removed (fallback for strict accelerate).
    Also enables low_cpu_mem_usage and tries the memory-efficient "sdpa"
    attention implementation when available.
    """
    kw.setdefault("low_cpu_mem_usage", True)
    torch.cuda.empty_cache()
    gc.collect()

    variants = [dict(kw)]

    sdpa_kw = dict(kw)
    sdpa_kw.setdefault("attn_implementation", "sdpa")
    if sdpa_kw != variants[0]:
        variants.append(sdpa_kw)

    renamed = dict(sdpa_kw)
    if "torch_dtype" in renamed:
        renamed["dtype"] = renamed.pop("torch_dtype")
    if renamed != variants[-1]:
        variants.append(renamed)

    if "max_memory" in renamed:
        no_mm = dict(renamed)
        no_mm.pop("max_memory", None)
        variants.append(no_mm)

    last_exc = None
    for i, variant in enumerate(variants):
        try:
            return cls.from_pretrained(name, **variant)
        except (TypeError, ValueError) as exc:
            last_exc = exc
            msg = str(exc)
            relevant = any(t in msg for t in
                           ("torch_dtype", "dtype", "max_memory", "Device", "not recognized",
                            "attn_implementation"))
            if i < len(variants) - 1 and relevant:
                print(f"WARNING: retry {i + 1}/{len(variants)} for {name}: {msg[:170]}")
                continue
            raise
    raise last_exc


def load_embedder():
    tok = AutoTokenizer.from_pretrained(CONFIG["embedding_model"], padding_side="left")
    mdl = load_causal(
        AutoModel, CONFIG["embedding_model"],
        torch_dtype=torch.float16,
        device_map="auto",
        max_memory=resolve_max_memory(),
    )
    mdl.eval()
    return tok, mdl


# make_embeddings is the original Phase-1 function loaded in §1.
def embed_texts(texts, tokenizer, model, batch_size=4, max_length=1024, desc=""):
    globals()["tokenizer"] = tokenizer
    globals()["model"] = model
    t0 = time.time()
    out = make_embeddings(texts, batch_size=batch_size, max_length=max_length)
    if desc:
        print(f"  {desc}: {len(texts)} texts in {time.time() - t0:.0f}s", flush=True)
    return out


def align_embeddings(ids, emb, target_ids):
    """Reorder embedding rows to match target_ids and assert full coverage."""
    idx = {d: i for i, d in enumerate(ids)}
    missing = [d for d in target_ids if d not in idx]
    assert not missing, f"{len(missing)} doc ids missing in cached embeddings, e.g. {missing[:3]}"
    order = np.array([idx[d] for d in target_ids])
    return emb[order]


# ------------------------------------------------------------------ passages
import hashlib

# Cache metadata binds generated embeddings to corpus contents and Phase-1 code.
_hash = hashlib.sha256()
for _id, _text in zip(DOC_IDS, PASSAGE_TEXTS):
    _hash.update(json.dumps([_id, _text], ensure_ascii=False).encode("utf-8"))
    _hash.update(b"\n")
_current_meta = {
    "corpus_sha256": _hash.hexdigest(),
    "phase1_source_sha256": hashlib.sha256(phase1_source.encode("utf-8")).hexdigest(),
    "embedding_model": CONFIG["embedding_model"],
    "max_length": CONFIG["embed_max_len"],
    "pooling": "last_token_l2_normalized",
}

def _read_passage_embeddings(path):
    with np.load(path, allow_pickle=False) as z:
        ids = [str(x) for x in z["doc_ids"]]
        emb = z["embeddings"]
        if (emb.ndim != 2 or emb.shape[0] != len(ids) or len(ids) != len(set(ids))
                or set(ids) != set(DOC_IDS) or not np.isfinite(emb).all()):
            raise ValueError("Embedding matrix or document IDs do not match the current corpus.")
        norms = np.linalg.norm(emb.astype(np.float32), axis=1)
        if not np.allclose(norms, 1.0, atol=1e-3):
            raise ValueError("Passage embeddings must be L2-normalized.")
        return align_embeddings(ids, emb, DOC_IDS).astype(np.float32, copy=False)

embedding_path = PHASE1_ARTIFACTS["document_embeddings_4B.npz"]
metadata_path = embedding_path.with_name("document_embeddings_4B.meta.json")
if not metadata_path.is_file():
    raise FileNotFoundError(f"Required embedding metadata not found: {metadata_path}")
try:
    actual_meta = json.loads(metadata_path.read_text(encoding="utf-8"))
    mismatches = {key: {"expected": value, "found": actual_meta.get(key)}
                  for key, value in _current_meta.items() if actual_meta.get(key) != value}
    if mismatches:
        raise ValueError(f"Embedding metadata mismatch: {mismatches}")
    passage_emb = _read_passage_embeddings(embedding_path)
except Exception as exc:
    raise RuntimeError(f"Cannot load valid document embeddings {embedding_path}: {exc}") from exc
PASSAGE_EMB_ORIGIN = f"Dataset file: {embedding_path}"
print("Loaded verified document embedding:", embedding_path)
print("Metadata is not proof that the matrix was generated by the stated model.")

_query_meta_path = ARTIFACT_DIR / "query_embeddings.meta.json"
_sent_meta_path = ARTIFACT_DIR / "sentence_embeddings.meta.json"
_query_meta = dict(_current_meta, query_sha256=hashlib.sha256(
    json.dumps([questions_dev, questions_hidden, QUERY_INSTRUCT], ensure_ascii=False, sort_keys=True).encode("utf-8")
).hexdigest())
_sent_meta = dict(_current_meta, sentence_max_length=CONFIG["sent_max_len"],
                  sentence_sha256=hashlib.sha256(json.dumps(DOC_SENTS, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest())
def _valid_metadata(meta_path, expected):
    try:
        return json.loads(meta_path.read_text(encoding="utf-8")) == expected
    except (OSError, ValueError):
        return False
need_sentences = CONFIG["run_sentence"] and not (
    (ARTIFACT_DIR / "sentence_embeddings.npz").is_file() and _valid_metadata(_sent_meta_path, _sent_meta))
have_query_cache = ((ARTIFACT_DIR / "query_embeddings.npz").is_file()
                    and _valid_metadata(_query_meta_path, _query_meta))

if need_sentences or not have_query_cache:
    print("\nloading embedding model:", CONFIG["embedding_model"])
    emb_tok, emb_model = load_embedder()

    # Reuse the query cache whenever it is present.
    if not have_query_cache:
        dev_q_emb = embed_texts(query_texts_for(questions_dev), emb_tok, emb_model,
                                    batch_size=CONFIG["embed_batch"],
                                    max_length=CONFIG["embed_max_len"], desc="dev queries")
        hid_q_emb = embed_texts(query_texts_for(questions_hidden), emb_tok, emb_model,
                                    batch_size=CONFIG["embed_batch"],
                                    max_length=CONFIG["embed_max_len"], desc="hidden queries")
        np.savez_compressed(ARTIFACT_DIR / "query_embeddings.npz",
                            dev_qids=np.asarray([str(q["qid"]) for q in questions_dev]),
                            dev=dev_q_emb,
                            hidden_qids=np.asarray([str(q["qid"]) for q in questions_hidden]),
                            hidden=hid_q_emb)
        _query_meta_path.write_text(json.dumps(_query_meta, indent=2), encoding="utf-8")
    
    # sentences
    if need_sentences:
        print("computing sentence embeddings ...")
        sent_rows, sent_texts = [], []
        for d in DOC_IDS:
            for si, s in enumerate(DOC_SENTS.get(d, [])):
                if s and s.strip():
                    sent_rows.append((d, si))
                    sent_texts.append(f"Title: {DOC_TITLE.get(d, '')}\nSentence: {s.strip()}")
        sent_emb = embed_texts(sent_texts, emb_tok, emb_model,
                                   batch_size=CONFIG["sent_batch"],
                                   max_length=CONFIG["sent_max_len"], desc="sentences")
        np.savez_compressed(ARTIFACT_DIR / "sentence_embeddings.npz",
                            doc_ids=np.asarray([r[0] for r in sent_rows]),
                            sent_idx=np.asarray([r[1] for r in sent_rows], dtype=np.int32),
                            embeddings=sent_emb)
        _sent_meta_path.write_text(json.dumps(_sent_meta, indent=2), encoding="utf-8")
        print("sentence embeddings:", sent_emb.shape)

    # The imported Phase-1 function retains tokenizer/model in its globals.
    globals().pop("tokenizer", None)
    globals().pop("model", None)
    del emb_tok, emb_model
    gc.collect()
    torch.cuda.empty_cache()
    print("embedding model unloaded")
else:
    print("all embeddings are cached; embedding model not needed")

# ------------------------------------------------------------------ load from cache
PASSAGE_EMB = passage_emb
assert PASSAGE_EMB.shape[0] == len(DOC_IDS)
assert np.isfinite(PASSAGE_EMB).all()

_z = np.load(ARTIFACT_DIR / "query_embeddings.npz", allow_pickle=False)
assert [str(x) for x in _z["dev_qids"]] == [str(q["qid"]) for q in questions_dev]
assert [str(x) for x in _z["hidden_qids"]] == [str(q["qid"]) for q in questions_hidden]
DEV_Q_EMB = _z["dev"].astype(np.float32)
HID_Q_EMB = _z["hidden"].astype(np.float32)
if (DEV_Q_EMB.ndim != 2 or HID_Q_EMB.ndim != 2 or
        DEV_Q_EMB.shape[1] != PASSAGE_EMB.shape[1] or HID_Q_EMB.shape[1] != PASSAGE_EMB.shape[1] or
        not np.isfinite(DEV_Q_EMB).all() or not np.isfinite(HID_Q_EMB).all()):
    raise ValueError("Query embeddings are invalid or incompatible with passage embeddings.")

if CONFIG["run_sentence"]:
    _s = np.load(ARTIFACT_DIR / "sentence_embeddings.npz", allow_pickle=False)
    SENT_DOC_IDS = [str(x) for x in _s["doc_ids"]]
    SENT_EMB     = _s["embeddings"].astype(np.float32)
    SENT_DOC_ROW = np.array([DOC_IDX[d] for d in SENT_DOC_IDS], dtype=np.int32)
    print(f"sentence matrix: {SENT_EMB.shape} ({len(SENT_DOC_IDS)} sentences)")

print("passage embeddings:", passage_emb.shape,
      "| dev queries:", DEV_Q_EMB.shape, "| hidden queries:", HID_Q_EMB.shape)


query prefix: 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: '
Loaded verified document embedding: /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/embeddings/document_embeddings_4B.npz
Metadata is not proof that the matrix was generated by the stated model.

loading embedding model: Qwen/Qwen3-Embedding-4B


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (4 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  dev queries: 300 texts in 7s
  hidden queries: 300 texts in 6s
computing sentence embeddings ...
  sentences: 14048 texts in 503s
sentence embeddings: (14048, 2560)
embedding model unloaded
sentence matrix: (14048, 2560) (14048 sentences)
passage embeddings: (3882, 2560) | dev queries: (300, 2560) | hidden queries: (300, 2560)


## §3 Hand-written BM25

Implements the standard BM25 scoring function:

\[
\text{score}(q,d)=\sum_{t\in q}\text{idf}(t)\cdot\frac{f_{td}\cdot(k_1+1)}{f_{td}+k_1\cdot(1-b+b\cdot|d|/\text{avgdl})}
\]

with \(k_1=1.5\), \(b=0.75\), and positive IDF defined as

\[
\text{idf}(t)=\ln\!\left(1+\frac{N-df_t+0.5}{df_t+0.5}\right).
\]

Properties:

- OOV terms are silently skipped (no crash on out-of-vocabulary words).
- Document masks (`exclude_ids`) are applied **before** `top-k`, so masked documents never appear in the returned list even when `k` is large relative to the remaining pool.
- Deterministic tie-break by ascending `doc_id`.


In [6]:
# =============================================================================
# §3  Hand-written BM25
# =============================================================================
TOKEN_RE = re.compile(r"[a-z0-9']+")

# A small, transparent English stop-word list.
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
    "has", "he", "in", "is", "it", "its", "of", "on", "that", "the",
    "to", "was", "were", "will", "with",
}


def tokenize(text, remove_stopwords=True):
    toks = TOKEN_RE.findall(str(text).lower())
    if remove_stopwords:
        toks = [t for t in toks if t not in STOPWORDS]
    return toks


class BM25Index:
    def __init__(self, doc_ids, texts, k1=1.5, b=0.75, remove_stopwords=True):
        self.doc_ids = [str(d) for d in doc_ids]
        self.k1 = float(k1)
        self.b = float(b)
        self.remove_stopwords = bool(remove_stopwords)
        self.N = len(self.doc_ids)

        tokens_per_doc = [tokenize(t, remove_stopwords) for t in texts]
        self.doc_len = np.array([len(toks) for toks in tokens_per_doc], dtype=np.float32)
        self.avgdl = float(self.doc_len.mean()) if self.N > 0 else 1.0

        # postings: term -> {doc_row: tf}
        df = Counter()
        postings_raw = defaultdict(dict)
        for i, toks in enumerate(tokens_per_doc):
            tf = Counter(toks)
            for term, count in tf.items():
                postings_raw[term][i] = count
            for term in tf.keys():
                df[term] += 1

        # positive IDF (BM25+ style variant used in the handout)
        self.idf = {term: math.log(1.0 + (self.N - c + 0.5) / (c + 0.5))
                    for term, c in df.items()}

        # Pre-compact postings into numpy arrays for fast scoring.
        self.postings = {}
        for term, row_tf in postings_raw.items():
            rows = np.array(sorted(row_tf.keys()), dtype=np.int64)
            tfs = np.array([row_tf[r] for r in rows], dtype=np.float32)
            self.postings[term] = (rows, tfs)

        # per-document normalization factor: k1 * (1 - b + b * |d| / avgdl)
        self.norm = self.k1 * (1.0 - self.b + self.b * self.doc_len / max(self.avgdl, 1e-9))

    def score(self, query):
        """Return an (N,) array of BM25 scores for all documents."""
        scores = np.zeros(self.N, dtype=np.float32)
        for term in tokenize(query, self.remove_stopwords):
            if term not in self.postings:
                continue                           # OOV term -> safely ignored
            rows, tfs = self.postings[term]
            idf = self.idf[term]
            # Standard BM25 term contribution.
            scores[rows] += idf * (tfs * (self.k1 + 1.0)) / (tfs + self.norm[rows])
        return scores

    def search(self, query, k=10, exclude_ids=frozenset()):
        """Return the top-k doc ids, with masked documents removed BEFORE top-k."""
        if k <= 0:
            raise ValueError("k must be positive")
        s = self.score(query)
        excluded = {str(x) for x in exclude_ids}
        excl = {i for i, d in enumerate(self.doc_ids) if str(d) in excluded}
        if excl:
            s = s.copy()
            s[list(excl)] = -np.inf
        k_eff = max(1, min(k, self.N - len(excl)))
        # Stable argsort on (-score, doc_id) for deterministic tie-breaking.
        order = np.lexsort((np.array(self.doc_ids), -s))
        return [self.doc_ids[i] for i in order[:k_eff] if np.isfinite(s[i])]

    def explain(self, query, doc_id):
        """Return per-term contributions for one document (used in error analysis)."""
        row = self.doc_ids.index(str(doc_id))
        out = {}
        for term in tokenize(query, self.remove_stopwords):
            if term not in self.postings:
                out[term] = 0.0
                continue
            rows, tfs = self.postings[term]
            if row not in set(int(r) for r in rows):
                out[term] = 0.0
                continue
            j = list(int(r) for r in rows).index(row)
            tf = float(tfs[j])
            out[term] = float(self.idf[term] * tf * (self.k1 + 1.0) / (tf + self.norm[row]))
        return out


BM25 = BM25Index(DOC_IDS, PASSAGE_TEXTS,
                 k1=CONFIG["bm25_k1"], b=CONFIG["bm25_b"],
                 remove_stopwords=CONFIG["bm25_remove_stopwords"])

# -------------- self-tests
_top = BM25.search(questions_dev[0]["question"], k=10)
assert len(_top) == 10 and len(set(_top)) == 10
_excl = {_top[0]}
assert _top[0] not in BM25.search(questions_dev[0]["question"], k=10, exclude_ids=_excl)
# mask should never produce fewer than k results when enough unmasked docs exist
_mask_n = max(1, min(5, BM25.N - 10))
assert len(BM25.search(questions_dev[0]["question"], k=10,
                       exclude_ids=set(DOC_IDS[:_mask_n]))) == 10
# OOV query must not crash and should return all-zero scores effectively
_oov = BM25.search("zzzznonexistentword xyzqzz", k=5)
assert isinstance(_oov, list)
# empty query
assert isinstance(BM25.search("", k=5), list)
# k larger than N
assert len(BM25.search(questions_dev[0]["question"], k=BM25.N + 100)) <= BM25.N
# finite scores
_q = BM25.score(questions_dev[0]["question"])
assert np.isfinite(_q).all()
assert np.isfinite(list(BM25.idf.values())).all()
# mask everything
assert BM25.search(questions_dev[0]["question"], k=10, exclude_ids=set(DOC_IDS)) == []
print("BM25 self-checks passed. sample top-3:", _top[:3])

# Independent rank_bm25 validation; its default BM25Okapi IDF differs from the
# positive-IDF formula required by the handout, so align only the IDF variant.
try:
    from rank_bm25 import BM25Okapi
except ImportError as exc:
    raise ImportError("rank_bm25 is required for the reference self-check. Enable internet and install rank_bm25.") from exc
_reference_tokens = [tokenize(t, CONFIG["bm25_remove_stopwords"]) for t in PASSAGE_TEXTS]
_reference = BM25Okapi(_reference_tokens, k1=CONFIG["bm25_k1"], b=CONFIG["bm25_b"])
_raw_idf_reference = dict(_reference.idf)
_ref_df = Counter(term for doc in _reference_tokens for term in set(doc))
_reference.idf = {term: math.log1p((len(_reference_tokens) - df + 0.5) / (df + 0.5))
                  for term, df in _ref_df.items()}
_validation_queries = [q["question"] for q in questions_dev[:min(50, len(questions_dev))]]
_validation_queries += ["", "zzzznonexistentword xyzqzz", "film film director"]
_max_abs = 0.0
for _query in _validation_queries:
    _got = BM25.score(_query)
    _expected = np.asarray(_reference.get_scores(tokenize(_query, CONFIG["bm25_remove_stopwords"])), dtype=np.float32)
    _diff = float(np.max(np.abs(_got - _expected))) if len(_got) else 0.0
    _max_abs = max(_max_abs, _diff)
    if not np.allclose(_got, _expected, rtol=1e-5, atol=1e-5):
        raise AssertionError(f"BM25 score mismatch for query {_query!r}: max abs difference {_diff}")
    # Compare scores for a masked set as well; ranking of very close values is
    # separately affected by float32 ties and the deterministic ID tie-break.
    _masked = set(DOC_IDS[:min(5, max(0, len(DOC_IDS) - 10))])
    assert not (set(BM25.search(_query, k=min(10, len(DOC_IDS)), exclude_ids=_masked)) & _masked)
_bm25_check = {
    "library": "rank_bm25.BM25Okapi", "queries_tested": len(_validation_queries),
    "max_abs_score_diff": _max_abs, "reference_idf": "positive handout log1p variant",
    "rank_bm25_default_idf_note": "The library's default IDF differs; the reference IDF was aligned to the handout formula.",
    "default_idf_different_terms": sum(not np.isclose(_raw_idf_reference[t], _reference.idf[t]) for t in _reference.idf),
    "vocabulary_size": len(_reference.idf), "score_match": True,
}
(ARTIFACT_DIR / "bm25_reference_check.json").write_text(json.dumps(_bm25_check, indent=2), encoding="utf-8")
print("BM25 vs rank_bm25 reference check:", _bm25_check)


BM25 self-checks passed. sample top-3: ['p02917', 'p03320', 'p02916']
BM25 vs rank_bm25 reference check: {'library': 'rank_bm25.BM25Okapi', 'queries_tested': 53, 'max_abs_score_diff': 3.814697265625e-06, 'reference_idf': 'positive handout log1p variant', 'rank_bm25_default_idf_note': "The library's default IDF differs; the reference IDF was aligned to the handout formula.", 'default_idf_different_terms': 28034, 'vocabulary_size': 28034, 'score_match': True}


## §4 Reciprocal Rank Fusion

Fuses multiple ranked lists into a single ranking using

\[
\text{RRF}(d)=\sum_{r}\frac{1}{k+\text{rank}_r(d)}, \quad k=60,
\]

where `rank` starts at **1** (not 0). Documents that do not appear in a particular list contribute nothing from that source. Ties are broken deterministically by ascending `doc_id`. Masked documents (`exclude_ids`) are dropped before aggregation.


In [7]:
# =============================================================================
# §4  Reciprocal Rank Fusion
# =============================================================================
def rrf_fuse(rank_lists, rrf_k=60, top_k=10, exclude_ids=frozenset()):
    """Fuse multiple ranked lists via RRF:  score(d) = Σ 1/(rrf_k + rank_r(d)).

    rank starts at 1. Documents not present in a list receive zero contribution
    from that source. Tie-break is deterministic (ascending doc id).
    """
    assert rrf_k >= 0, "rrf_k must be >= 0"
    excl = set(str(x) for x in exclude_ids)
    scores = defaultdict(float)
    for lst in rank_lists:
        for rank, d in enumerate(lst, start=1):
            d = str(d)
            if d in excl:
                continue
            scores[d] += 1.0 / (rrf_k + rank)
    ordered = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))
    return [d for d, _ in ordered[:top_k]]


def rrf_scores(rank_lists, rrf_k=60, exclude_ids=frozenset()):
    """Return the full RRF score dict (for reporting/analysis)."""
    excl = set(str(x) for x in exclude_ids)
    scores = defaultdict(float)
    for lst in rank_lists:
        for rank, d in enumerate(lst, start=1):
            d = str(d)
            if d in excl:
                continue
            scores[d] += 1.0 / (rrf_k + rank)
    return dict(scores)


# -------------- self-tests
_a = ["p001", "p002", "p003", "p004"]
_b = ["p002", "p001", "p004", "p003"]
# Tie between p001 and p002 (both appear once at rank 1 and once at rank 2) -> smaller id wins
assert rrf_fuse([_a, _b], rrf_k=60, top_k=4) == ["p001", "p002", "p003", "p004"] or \
       rrf_fuse([_a, _b], rrf_k=60, top_k=4)[:2] == ["p001", "p002"]
assert len(rrf_fuse([_a, _b], rrf_k=60, top_k=4)) == 4
# rank 1 -> 1/61
_pts = rrf_scores([["pA"]], rrf_k=60)
assert abs(_pts["pA"] - 1.0 / 61.0) < 1e-9
# rrf_k=0 -> harmonic-style 1/rank
_pts0 = rrf_scores([["pA", "pB"], ["pA"]], rrf_k=0)
assert abs(_pts0["pA"] - (1.0 + 1.0)) < 1e-9
# pB only in list 2 at rank 3 -> 1/63
_mix = rrf_scores([["pA", "pB"], ["pA", "pC", "pB"]], rrf_k=60)
assert abs(_mix["pB"] - (1.0 / 62.0 + 1.0 / 63.0)) < 1e-9
# excluded id must not appear
assert "p001" not in rrf_fuse([_a, _b], rrf_k=60, top_k=4, exclude_ids={"p001"})
# deterministic: same input -> same output across two calls
assert rrf_fuse([_a, _b]) == rrf_fuse([_a, _b])
print("RRF self-checks passed")


RRF self-checks passed


## §5 Dense and sentence-level retrieval

Because the corpus is small (3,882 passages), dense retrieval uses an **exact** full-matrix dot product between the query vector and all passage vectors; since both are L2-normalized this equals cosine similarity.

For the sentence-level experiment:

1. Every sentence in every passage is embedded individually (with a small `Title:` + `Sentence:` prefix).
2. Sentence similarities are **max-pooled** per passage (`np.maximum.at`), so each passage receives the score of its best-matching sentence.
3. Retrieval still returns passage IDs, so downstream context packing / LLM code stays unchanged.


In [8]:
# =============================================================================
# §5  Exact dense search and sentence-level max-pool retrieval
# =============================================================================
# Adapt the original Phase-1 dense_search to Phase-2 score arrays.
topk_from_scores = dense_search_from_scores


def dense_scores(q_vec, exclude_ids=frozenset()):
    """Cosine similarity of a query vector against all passages.

    Since both matrices are L2-normalized, the dot product equals cosine.
    """
    sim = PASSAGE_EMB @ np.asarray(q_vec, dtype=np.float32)
    return sim


def dense_search(q_vec, k=10, exclude_ids=frozenset()):
    return topk_from_scores(dense_scores(q_vec, exclude_ids), DOC_IDS, k=k, exclude_ids=exclude_ids)


def sentence_scores(q_vec, exclude_ids=frozenset()):
    """Per-passage score defined as the MAX similarity over its sentences."""
    sims = SENT_EMB @ np.asarray(q_vec, dtype=np.float32)
    pass_scores = np.full(len(DOC_IDS), -np.inf, dtype=np.float32)
    np.maximum.at(pass_scores, SENT_DOC_ROW, sims)
    excl = {DOC_IDX[d] for d in exclude_ids if d in DOC_IDX}
    if excl:
        pass_scores[list(excl)] = -np.inf
    return pass_scores


def sentence_search(q_vec, k=10, exclude_ids=frozenset()):
    return topk_from_scores(sentence_scores(q_vec, exclude_ids), DOC_IDS, k=k, exclude_ids=exclude_ids)


# -------------- self-tests
_q0 = DEV_Q_EMB[0]
_dtop = dense_search(_q0, k=10)
assert len(_dtop) == 10 and len(set(_dtop)) == 10
assert _dtop[0] in DOC_IDX
_excl = {_dtop[0]}
assert _dtop[0] not in dense_search(_q0, k=10, exclude_ids=_excl)
assert dense_scores(_q0).max() == dense_scores(_q0)[DOC_IDX[_dtop[0]]]
if CONFIG["run_sentence"]:
    _s = sentence_search(_q0, k=10)
    assert len(_s) == 10 and len(set(_s)) == 10
print("Dense search self-checks passed. dense top-3:", _dtop[:3])


Dense search self-checks passed. dense top-3: ['p02917', 'p02916', 'p03365']


## §7 Building ranking tables

For every question (dev and hidden) we build ranked lists for the active systems:

| System | Description |
|---|---|
| `dense` | Pure dense retrieval baseline. |
| `bm25` | Pure BM25 baseline. |
| `hybrid` | RRF fusion of `dense` + `bm25`. |
| `hybrid_sentence` | RRF fusion of sentence-level dense + passage BM25. |
| `hybrid_goldmask` | Hybrid retrieval with gold documents **masked out** (abstention experiment). |

Each returned list always contains exactly 10 unique valid doc IDs, even when masks are applied.


In [9]:
# =============================================================================
# §7  Build ranking tables for dev and hidden splits
# =============================================================================
def gold_ids_of(qid):
    lab = LABELS_BY_QID.get(str(qid))
    return [str(x) for x in (lab["gold_doc_ids"] if lab else [])]


def build_rankings(items, q_emb, with_goldmask=False, tag=""):
    """For each question, build per-system rankings: dense/bm25/hybrid + (optional)
    sentence and goldmask.

    Returns: dict {system: {qid: [doc_ids, ...]}}
    """
    systems = ["dense", "bm25", "hybrid"]
    if CONFIG["run_sentence"]:
        systems.append("hybrid_sentence")
    if with_goldmask and CONFIG["run_goldmask"]:
        systems.append("hybrid_goldmask")

    out = {s: {} for s in systems}
    t0 = time.time()

    for i, item in enumerate(items):
        qid, qtext, vec = str(item["qid"]), item["question"], q_emb[i]

        # Standard runs.
        d_list = dense_search(vec, CONFIG["fusion_depth"])
        b_list = BM25.search(qtext, CONFIG["fusion_depth"])
        h_list = rrf_fuse([d_list, b_list], CONFIG["rrf_k"], CONFIG["k_output"])

        out["dense"][qid] = d_list[:CONFIG["k_output"]]
        out["bm25"][qid] = b_list[:CONFIG["k_output"]]
        out["hybrid"][qid] = h_list

        if CONFIG["run_sentence"]:
            s_list = sentence_search(vec, CONFIG["fusion_depth"])
            out["hybrid_sentence"][qid] = rrf_fuse(
                [s_list, b_list], CONFIG["rrf_k"], CONFIG["k_output"])

        # Gold-mask abstention experiment: mask out all gold documents.
        if with_goldmask and CONFIG["run_goldmask"]:
            excl = frozenset(gold_ids_of(qid))
            d_m = dense_search(vec, CONFIG["fusion_depth"], excl)
            b_m = BM25.search(qtext, CONFIG["fusion_depth"], excl)
            out["hybrid_goldmask"][qid] = rrf_fuse(
                [d_m, b_m], CONFIG["rrf_k"], CONFIG["k_output"], excl)

        if (i + 1) % 50 == 0 or i + 1 == len(items):
            print(f"  [{tag}] {i + 1}/{len(items)} questions  "
                  f"({time.time() - t0:.0f}s)", flush=True)

    # Structural guarantees.
    for s, table in out.items():
        assert len(table) == len(items), f"{s}: {len(table)} rows for {len(items)} questions"
        for qid, ids in table.items():
            assert len(ids) == CONFIG["k_output"], f"{s}/{qid}: expected {CONFIG['k_output']} ids, got {len(ids)}"
            assert len(set(ids)) == len(ids), f"{s}/{qid}: duplicate ids"
            assert all(d in DOC_IDX for d in ids), f"{s}/{qid}: unknown doc id"
            if with_goldmask and s == "hybrid_goldmask":
                assert not (set(ids) & set(gold_ids_of(qid))), f"{qid}: gold doc leaked into goldmask output"

    return out


DEV_ITEMS = questions_dev if not CONFIG["dev_limit"] else questions_dev[:CONFIG["dev_limit"]]
DEV_EMB = DEV_Q_EMB if not CONFIG["dev_limit"] else DEV_Q_EMB[:CONFIG["dev_limit"]]

print(f"building rankings: dev={len(DEV_ITEMS)} questions, "
      f"hidden={len(questions_hidden)} questions")
RANKINGS = {}
if CONFIG["run_dev"]:
    RANKINGS["dev"] = build_rankings(DEV_ITEMS, DEV_EMB, with_goldmask=True, tag="dev")
    # Compare the regenerated dense rankings with the saved Phase-1 retrievals.
    # Dense retrieval is still required for hybrid RRF, but no dense answers are regenerated.
    _p1_rows = read_jsonl(PHASE1_ARTIFACTS["NLP_phase1_output/dev/dense/predictions.jsonl"])
    _p1_by_qid = {str(r["qid"]): r for r in _p1_rows}
    if len(_p1_by_qid) != len(_p1_rows):
        raise ValueError("Duplicate qids in the saved Phase-1 dense predictions.")
    _pairs = []
    for _item in DEV_ITEMS:
        _qid = str(_item["qid"])
        if _qid not in _p1_by_qid:
            raise ValueError(f"Missing Phase-1 dense prediction for {_qid}")
        _old = [str(d) for d in _p1_by_qid[_qid]["retrieved"][:CONFIG["k_output"]]]
        _new = RANKINGS["dev"]["dense"][_qid]
        if len(_old) != CONFIG["k_output"] or len(set(_old)) != len(_old) or not set(_old) <= set(DOC_IDS):
            raise ValueError(f"Invalid Phase-1 dense ranking for {_qid}")
        _pairs.append({"qid": _qid, "same_order": _old == _new,
                       "same_set": set(_old) == set(_new),
                       "overlap_at_10": len(set(_old) & set(_new))})
    DENSE_RANK_AUDIT = {"n": len(_pairs),
        "exact_order_percent": 100 * sum(x["same_order"] for x in _pairs) / len(_pairs),
        "exact_set_percent": 100 * sum(x["same_set"] for x in _pairs) / len(_pairs),
        "mean_overlap_at_10": sum(x["overlap_at_10"] for x in _pairs) / len(_pairs),
        "rows": _pairs}
    (ARTIFACT_DIR / "dense_ranking_audit.json").write_text(json.dumps(DENSE_RANK_AUDIT, indent=2), encoding="utf-8")
    print("Saved vs regenerated Phase-1 dense retrieval:", {k: v for k, v in DENSE_RANK_AUDIT.items() if k != "rows"})
RANKINGS["hidden"] = build_rankings(questions_hidden, HID_Q_EMB, with_goldmask=False, tag="hidden")

for split, table in RANKINGS.items():
    path = ARTIFACT_DIR / f"rankings_{split}.json"
    path.write_text(json.dumps(table, ensure_ascii=False), encoding="utf-8")
    print(f"saved {path} | systems: {list(table)}")

print("\nsample (dev, first question):")
_q0 = str(DEV_ITEMS[0]["qid"])
print(" question:", DEV_ITEMS[0]["question"])
print(" gold    :", gold_ids_of(_q0))
for s in RANKINGS["dev"]:
    print(f" {s:16s}: {RANKINGS['dev'][s][_q0][:6]}")


building rankings: dev=300 questions, hidden=300 questions
  [dev] 50/300 questions  (1s)
  [dev] 100/300 questions  (2s)
  [dev] 150/300 questions  (3s)
  [dev] 200/300 questions  (3s)
  [dev] 250/300 questions  (4s)
  [dev] 300/300 questions  (5s)
Saved vs regenerated Phase-1 dense retrieval: {'n': 300, 'exact_order_percent': 100.0, 'exact_set_percent': 100.0, 'mean_overlap_at_10': 10.0}
  [hidden] 50/300 questions  (1s)
  [hidden] 100/300 questions  (1s)
  [hidden] 150/300 questions  (2s)
  [hidden] 200/300 questions  (2s)
  [hidden] 250/300 questions  (3s)
  [hidden] 300/300 questions  (4s)
saved /kaggle/working/NLP_phase2/artifacts/rankings_dev.json | systems: ['dense', 'bm25', 'hybrid', 'hybrid_sentence', 'hybrid_goldmask']
saved /kaggle/working/NLP_phase2/artifacts/rankings_hidden.json | systems: ['dense', 'bm25', 'hybrid', 'hybrid_sentence']

sample (dev, first question):
 question: Did the movies Ride A Crooked Trail and The Glassworker, originate from the same country?
 gold 

## §8 Context packing

Documents are packed in rank order. The complete candidate context, including IDs, titles and separators, is counted with the LLM tokenizer. Packing stops before the budget is exceeded; full prompt length is checked separately. `retrieved` retains the original ranking.


In [10]:
# =============================================================================
# §8  Context packing under an LLM-token budget
# =============================================================================
llm_tokenizer = AutoTokenizer.from_pretrained(CONFIG["llm_model"], padding_side="left")
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token
print("LLM tokenizer:", CONFIG["llm_model"], "| pad:", llm_tokenizer.pad_token)

DOC_BLOCK = {d: f"[{d}] Title: {DOC_TITLE.get(d, '')}\n{DOC_TEXT.get(d, '')}" for d in DOC_IDS}
BLOCK_SEP_TOKENS = len(llm_tokenizer.encode("\n\n", add_special_tokens=False)) or 1
print("separator token count (LLM tokenizer):", BLOCK_SEP_TOKENS)

# Pre-compute (and cache) the token count of every passage block using the LLM tokenizer.
_tok_cache = ARTIFACT_DIR / "passage_token_counts.json"
if _tok_cache.exists():
    DOC_TOKENS = {k: int(v) for k, v in json.loads(_tok_cache.read_text()).items()}
    print(f"loaded token counts for {len(DOC_TOKENS)} blocks")
else:
    t0 = time.time()
    enc = llm_tokenizer([DOC_BLOCK[d] for d in DOC_IDS], add_special_tokens=False)
    DOC_TOKENS = {d: len(ids) for d, ids in zip(DOC_IDS, enc["input_ids"])}
    _tok_cache.write_text(json.dumps(DOC_TOKENS))
    print(f"tokenized {len(DOC_TOKENS)} blocks in {time.time() - t0:.1f}s")

_tok_vals = np.array(list(DOC_TOKENS.values()))
print(f"passage block tokens: mean={_tok_vals.mean():.0f} max={_tok_vals.max()} "
      f"p95={np.percentile(_tok_vals, 95):.0f}")


def pack_context(doc_ids, budget=None, k_context=None, policy="stop"):
    """Greedy packing of passages in rank order up to `budget` LLM tokens.

    Parameters
    ----------
    doc_ids : iterable of doc ids (in rank order, best first).
    budget  : max tokens for the concatenated document blocks (default from CONFIG).
    k_context : max number of passages considered.
    policy  : "stop" -> break at the first overflow; "skip" -> drop the overflowing
              passage and keep going.

    Returns a dict with: ids, text, n_tokens, dropped, skipped.
    """
    budget = int(budget if budget is not None else CONFIG["max_context_tokens"])
    k_context = int(k_context if k_context is not None else CONFIG["k_context"])
    assert policy in ("stop", "skip"), "policy must be 'stop' or 'skip'"

    chosen, used, blocks, skipped = [], 0, [], []
    for d in list(doc_ids)[:k_context]:
        candidate_text = "\n\n".join(blocks + [DOC_BLOCK[d]])
        candidate_tokens = len(llm_tokenizer.encode(candidate_text, add_special_tokens=False))
        if candidate_tokens > budget:
            if policy == "stop":
                break
            skipped.append(d)
            continue
        chosen.append(d)
        used = candidate_tokens
        blocks.append(DOC_BLOCK[d])

    dropped = [d for d in list(doc_ids)[:k_context] if d not in chosen]
    return {
        "ids": chosen,
        "text": "\n\n".join(blocks),
        "n_tokens": used,
        "dropped": dropped,
        "skipped": skipped,
    }


# -------------- self-tests
_pick = RANKINGS["dev"]["hybrid"][str(DEV_ITEMS[0]["qid"])]
_pk = pack_context(_pick, budget=10**6)
assert _pk["ids"] == _pick, "rank order must be preserved"
assert _pk["n_tokens"] == len(llm_tokenizer.encode(_pk["text"], add_special_tokens=False))

_pk_def = pack_context(_pick)
assert _pk_def["n_tokens"] <= CONFIG["max_context_tokens"]

_tight = pack_context(_pick, budget=60)
assert _tight["n_tokens"] <= 60, f"budget violated: {_tight['n_tokens']}"
assert len(_tight["ids"]) < len(_pick), "tight budget must drop passages"

_skip = pack_context(_pick, budget=60, policy="skip")
assert _skip["n_tokens"] <= 60
assert len(_skip["skipped"]) > 0 or len(_skip["ids"]) < 2

_empty = pack_context([], budget=100)
assert _empty["ids"] == [] and _empty["text"] == "" and _empty["n_tokens"] == 0

assert pack_context(_pick, budget=0)["ids"] == []

print("\npacking self-checks passed.")
print(f"sample: {len(_pk_def['ids'])} docs, {_pk_def['n_tokens']} tokens "
      f"(budget {CONFIG['max_context_tokens']})")
print(f"tight : {len(_tight['ids'])} docs, {_tight['n_tokens']} tokens (budget 60)")


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

LLM tokenizer: Qwen/Qwen3-8B | pad: <|endoftext|>
separator token count (LLM tokenizer): 1
tokenized 3882 blocks in 0.5s
passage block tokens: mean=127 max=1255 p95=371

packing self-checks passed.
sample: 10 docs, 757 tokens (budget 1800)
tight : 0 docs, 0 tokens (budget 60)


## §9 Language model and answer parsing

The generator (`Qwen/Qwen3-8B`) is prompted with a strict template that enforces:

- A short (1–6 word) final answer on the very last line: `Answer: <short answer>`.
- Exact `Answer: Yes` / `Answer: No` for boolean questions.
- Inline citations in square brackets using **only** the document IDs that appear in the `Documents:` block.
- `INSUFFICIENT_INFORMATION` (and no `Answer:` line) when evidence is missing.

A dedicated parser strips any `<think>…</think>` blocks, extracts citations, finds the last `Answer:` line, normalizes the answer text, maps pure-ID answers to document titles, and treats empty / unknown answers as abstentions.


In [11]:
# =============================================================================
# §9  Language model, prompt, batched generation, output parser
# =============================================================================
SYSTEM_PROMPT = ("You are a precise question-answering system. You answer only from the "
                 "provided documents, cite them, and admit when the evidence is missing.")

USER_TEMPLATE = """Answer the question using ONLY the documents below. Do not use outside knowledge.

Rules:
1. You may reason briefly, but the VERY LAST line of your reply must be exactly:
Answer: <short answer>
2. <short answer> is 1-6 words: a title, a name, a place, a date, a number, or Yes/No. Never a full sentence.
3. For yes/no questions write exactly "Answer: Yes" or "Answer: No".
4. Cite every supporting document inline with its id in square brackets, e.g. [p00012].
   Use only ids that appear in the Documents block. Never invent ids.
5. If the documents do not contain enough information to answer, reply with exactly:
INSUFFICIENT_INFORMATION
and do not write an Answer line.
6. If several candidates seem plausible, commit to exactly one. Never write "X or Y".

Documents:
{context}

Question: {question}

Answer:
"""

PROMPT_FILE = PROMPT_DIR / "phase2_strict.txt"
PROMPT_FILE.write_text(SYSTEM_PROMPT + "\n\n" + USER_TEMPLATE, encoding="utf-8")
print("prompt saved to:", PROMPT_FILE)

# ---------------------------------------------------------------- load LLM
torch.cuda.empty_cache()
gc.collect()
print("loading LLM:", CONFIG["llm_model"])
for i in range(torch.cuda.device_count()):
    free_m, total_m = torch.cuda.mem_get_info(i)
    print(f"  GPU {i} free before LLM load: {free_m / 1024**3:.2f} GiB / {total_m / 1024**3:.2f} GiB")

llm_tokenizer = AutoTokenizer.from_pretrained(CONFIG["llm_model"], padding_side="left")
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

llm_model = load_causal(
    AutoModelForCausalLM, CONFIG["llm_model"],
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory=resolve_max_memory(),
)
llm_model.eval()
print("device map:", getattr(llm_model, "hf_device_map", "single device"))
for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i) / 1024**3
    free_m, total_m = torch.cuda.mem_get_info(i)
    print(f"  GPU {i}: allocated {alloc:.2f} GiB | free {free_m / 1024**3:.2f} GiB")


def apply_chat(system, user):
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": user}]
    try:
        return llm_tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return llm_tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)


def _generate_once(prompts, max_new_tokens):
    """Single greedy generation pass (no retry). Caller handles OOM.

    Uses the Phase-2 batched greedy generation adapter. The original
    Phase-1 generator is single-sample and has a different prompt.
    """
    return generate_from_prompts(
        prompts, llm_tokenizer, llm_model,
        max_new_tokens=max_new_tokens,
        max_length=CONFIG["max_prompt_tokens"],
        pad_token_id=llm_tokenizer.pad_token_id)


@torch.inference_mode()
def generate_batch(prompts, max_new_tokens=None):
    """Greedy generation with automatic OOM fallback.

    Tries the full batch first; on OutOfMemoryError it clears the cache and
    retries by splitting the batch in half (recursively down to batch=1).
    Minor floating-point differences across batch sizes are possible.
    """
    max_new_tokens = int(max_new_tokens or CONFIG["max_new_tokens"])
    try:
        return _generate_once(prompts, max_new_tokens)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        if len(prompts) <= 1:
            raise
        mid = len(prompts) // 2
        print(f"  [OOM] batch {len(prompts)} -> retrying as {mid}+{len(prompts)-mid}",
              flush=True)
        return (generate_batch(prompts[:mid], max_new_tokens)
                + generate_batch(prompts[mid:], max_new_tokens))


# ---------------------------------------------------------------- output parser
ABSTAIN_TOKEN = "INSUFFICIENT_INFORMATION"
_ID_RE = re.compile(r"\bp\d{4,6}\b", flags=re.IGNORECASE)
_CITE_RE = re.compile(r"\[\s*(p\d{4,6})\s*\]", flags=re.IGNORECASE)


def strip_thinking(text):
    text = re.sub(r"<think(?:ing)?>.*?</think(?:ing)?>", " ", text, flags=re.DOTALL | re.IGNORECASE)
    # An unclosed thinking block is discarded rather than treated as an answer.
    text = re.sub(r"<think(?:ing)?>.*$", " ", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"</?think(?:ing)?>", " ", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()


def parse_llm_output(raw, context_ids):
    """Convert raw model output into a structured dict.

    Returns dict(answer, cited, abstained) where:
      - cited contains only ids that actually appear in the context.
      - abstained=True means the model refused / did not produce an answer.
      - A bare document id is not a valid answer.

    The answer text is extracted with the Phase-2 extract_short_answer adapter
    (cell 15) — no internal copy.
    """
    context_ids = [str(c) for c in context_ids]
    ctx_set = set(context_ids)
    text = strip_thinking(raw)

    cited = list(dict.fromkeys(m.lower() for m in _CITE_RE.findall(text) if m.lower() in ctx_set))
    # Accept JSON output as a fallback, without treating arbitrary IDs as citations.
    _json_text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.IGNORECASE)
    try:
        obj = json.loads(_json_text)
    except (ValueError, TypeError):
        obj = None
    if isinstance(obj, dict):
        if obj.get("abstained") is True:
            return {"answer": "", "cited": [], "abstained": True}
        value = obj.get("answer", "")
        answer = value.strip() if isinstance(value, str) else ""
        refs = obj.get("cited", obj.get("citations", []))
        if not isinstance(refs, list):
            refs = []
        cited = list(dict.fromkeys(str(c).lower().strip(" []") for c in refs
                                   if str(c).lower().strip(" []") in ctx_set))
    else:
        if ABSTAIN_TOKEN.lower() in text.lower():
            return {"answer": "", "cited": [], "abstained": True}
        answer = extract_short_answer(text).strip("\"'“”«»")
    if not answer:
        return {"answer": "", "cited": [], "abstained": True}

    if _ID_RE.fullmatch(answer.strip()) or answer.lower() in {"insufficient_information", "insufficient information", "unknown", "n a"}:
        return {"answer": "", "cited": [], "abstained": True}

    assert set(cited) <= ctx_set, "cited ids must come from the packed context"
    return {"answer": answer, "cited": cited, "abstained": False}


# -------------- parser self-tests
_ctx = ["p00001", "p00002"]
assert parse_llm_output("Answer: Yes", _ctx)["answer"] == "Yes"
assert parse_llm_output("blah blah\nAnswer: Paris [p00001]", _ctx) == \
    {"answer": "Paris", "cited": ["p00001"], "abstained": False}
assert parse_llm_output("Evidence [p00009] says X. Answer: X", _ctx)["cited"] == [], \
    "citations outside the context must be dropped"
assert parse_llm_output("INSUFFICIENT_INFORMATION", _ctx) == \
    {"answer": "", "cited": [], "abstained": True}
assert parse_llm_output("<think>long reasoning</think>Answer: No", _ctx)["answer"] == "No"
assert parse_llm_output("<think>unclosed reasoning", _ctx)["abstained"] is True
assert parse_llm_output("Answer: Paris [p00001] [p00001] [p00009]", _ctx)["cited"] == ["p00001"]
assert parse_llm_output("Answer: p00002", _ctx)["cited"] == []
assert parse_llm_output("Answer: p00002", _ctx)["abstained"] is True
assert parse_llm_output('{"answer":"Paris","cited":["p00001","p00009"],"abstained":false}', _ctx) == {"answer":"Paris", "cited":["p00001"], "abstained":False}
assert parse_llm_output('{"answer":"Paris","abstained":true}', _ctx)["abstained"] is True
assert parse_llm_output("Answer: ", _ctx)["abstained"] is True
assert parse_llm_output("Answer: The answer is \"Paris\".", _ctx)["answer"] == "Paris"
assert parse_llm_output("Answer: Paris [P00001]", _ctx)["answer"] == "Paris"
assert parse_llm_output("Answer: Paris [p1234]", ["p1234"])["answer"] == "Paris"
assert parse_llm_output("Answer: Paris [p123456]", ["p123456"])["answer"] == "Paris"
assert parse_llm_output('{"answer":"INSUFFICIENT_INFORMATION","abstained":false}', _ctx)["abstained"] is True
_id_widths = Counter(len(d) - 1 for d in DOC_IDS if re.fullmatch(r"p\d+", d))
print("Corpus p+digits ID widths:", dict(sorted(_id_widths.items())), "of", len(DOC_IDS), "documents")
if sum(_id_widths.get(n, 0) for n in range(4, 7)) != len(DOC_IDS):
    raise ValueError("Unsupported corpus doc_id format: expected p followed by 4 to 6 digits.")
print("parser self-checks passed")

# Quick smoke generation on the first dev question.
_smoke_q = questions_dev[0]["question"]
_smoke_ctx = pack_context(RANKINGS["dev"]["hybrid"][str(questions_dev[0]["qid"])])["ids"]
_smoke_prompt = apply_chat(SYSTEM_PROMPT, USER_TEMPLATE.format(
    context=pack_context(_smoke_ctx)["text"], question=_smoke_q))
_smoke_out = generate_batch([_smoke_prompt])[0]
print("\nsmoke generation:")
print("--- raw ---")
print(_smoke_out[:600])
print("--- parsed ---")
print(parse_llm_output(_smoke_out, _smoke_ctx))


prompt saved to: /kaggle/working/NLP_phase2/prompts/phase2_strict.txt
loading LLM: Qwen/Qwen3-8B
  GPU 0 free before LLM load: 14.39 GiB / 14.56 GiB
  GPU 1 free before LLM load: 14.39 GiB / 14.56 GiB
max_memory (int keys): {0: '12GiB', 1: '12GiB', 'cpu': '40GiB'}


model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 0, 'model.layers.17': 0, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}
  GPU 0: allocated 7.64 GiB | free 6.77 GiB
  GPU 1: allocated 7.64 GiB | free 6.77 GiB
Corpus p+digits ID widths: {5: 3882} o

## §10 Dev runs

Generate and save controlled dense, hybrid, sentence-level hybrid and gold-mask runs. Each has predictions and a configuration. The original Phase-1 dense predictions are reused for a historical comparison; controlled dense is generated under Phase-2 settings.


In [12]:
# =============================================================================
# §10  Run all dev systems, save predictions and run_config snapshots
# =============================================================================
import yaml

CONTEXT_USED = {}      # system -> {qid: [context ids actually packed]}
RAW_OUTPUTS = {}       # system -> {qid: raw LLM text}


def write_run_config(system, split, path, extra=None):
    cfg = {
        "system_name": system,
        "phase": 2,
        "passage_embeddings_source": PASSAGE_EMB_ORIGIN,
        "split": split,
        "description": {
            "dense": "Phase-1 dense retrieval recipe regenerated with Phase-2 prompt, packing and generation settings (controlled baseline)",
            "hybrid": "Dense + hand-written BM25 fused with RRF, context packing, citations, abstention",
            "hybrid_sentence": "Sentence-level dense retrieval (max-pool) fused with passage BM25 via RRF",
            "hybrid_goldmask": "Hybrid retrieval with gold documents masked (abstention experiment)",
        }.get(system, system),
        "models": {
            "llm": {"name": CONFIG["llm_model"], "where": "Kaggle, 2x Tesla T4, transformers fp16",
                    "temperature": CONFIG["temperature"]},
            "embedder": {"name": CONFIG["embedding_model"],
                         "max_sequence_length": CONFIG["embed_max_len"]},
        },
        "settings": {
            "retrieval": {
                "unit": "sentence" if system == "hybrid_sentence" else "passage",
                "dense_depth": CONFIG["fusion_depth"],
                "bm25_depth": CONFIG["fusion_depth"],
                "fusion": "rrf" if system.startswith("hybrid") else None,
                "rrf_k": CONFIG["rrf_k"],
                "bm25": {"k1": CONFIG["bm25_k1"], "b": CONFIG["bm25_b"],
                         "stopwords_removed": CONFIG["bm25_remove_stopwords"]},
                "output_k": CONFIG["k_output"],
                "context_k": CONFIG["k_context"],
                "document_mask": system == "hybrid_goldmask",
            },
            "context_packing": {"max_context_tokens": CONFIG["max_context_tokens"],
                                "policy": "stop", "token_counter": "llm_tokenizer"},
            "max_prompt_tokens": CONFIG["max_prompt_tokens"],
            "max_new_tokens": CONFIG["max_new_tokens"],
            "llm_batch": CONFIG["llm_batch"],
            "do_sample": False,
            "no_repeat_ngram_size": 8,
            "repetition_penalty": 1.05,
            "prompt_file": "prompts/phase2_strict.txt",
        },
        "command": "jupyter nbconvert --execute notebook.ipynb (sections §1-§15)",
        "date": str(date.today()),
    }
    if extra:
        cfg["settings"].update(extra)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")


def run_system(system, split, items, ranking_table, save=True, desc=""):
    """Generate answers for every question using the given ranking table."""
    records, ctx_map, raw_map = [], {}, {}
    t0 = time.time()
    n = len(items)
    batch = max(1, CONFIG["llm_batch"])

    for start in range(0, n, batch):
        chunk = items[start:start + batch]
        prompts, ctxs = [], []
        for it in chunk:
            qid = str(it["qid"])
            packed = pack_context(ranking_table[qid])
            ctxs.append(packed["ids"])
            prompt = apply_chat(SYSTEM_PROMPT, USER_TEMPLATE.format(
                context=packed["text"], question=it["question"]))
            if len(llm_tokenizer(prompt, add_special_tokens=True)["input_ids"]) > CONFIG["max_prompt_tokens"]:
                raise ValueError(f"Prompt for {qid} exceeds max_prompt_tokens; reduce the context budget.")
            prompts.append(prompt)
        raws = generate_batch(prompts)

        for it, raw, ctx in zip(chunk, raws, ctxs):
            qid = str(it["qid"])
            res = parse_llm_output(raw, ctx)
            records.append({
                "qid": qid,
                "answer": res["answer"],
                "retrieved": list(ranking_table[qid]),
                "cited": res["cited"],
                "abstained": res["abstained"],
            })
            ctx_map[qid] = ctx
            raw_map[qid] = raw

        done = start + len(chunk)
        if done % 50 < batch or done == n:
            rate = done / max(time.time() - t0, 1e-9)
            eta = (n - done) / max(rate, 1e-9)
            print(f"  [{system}] {done}/{n}  ({time.time() - t0:.0f}s, ETA {eta:.0f}s)",
                  flush=True)

    CONTEXT_USED[system] = ctx_map
    RAW_OUTPUTS[system] = raw_map

    if save:
        out_dir = RUNS_DIR / split / system
        save_predictions(records, out_dir / "predictions.jsonl")
        write_run_config(system, split, out_dir / "run_config.yaml")
        (ARTIFACT_DIR / f"context_used_{split}_{system}.json").write_text(
            json.dumps(ctx_map), encoding="utf-8")
        (ARTIFACT_DIR / f"raw_outputs_{split}_{system}.json").write_text(
            json.dumps(raw_map, ensure_ascii=False), encoding="utf-8")
    return records


# ---------------------------------------------------------------- dev runs
DEV_SYSTEMS = []
if CONFIG["run_dev"]:
    DEV_SYSTEMS.append("hybrid")
    if CONFIG["run_sentence"]:
        DEV_SYSTEMS.append("hybrid_sentence")
    if CONFIG["run_goldmask"]:
        DEV_SYSTEMS.append("hybrid_goldmask")

DEV_PREDS = {}
for system in DEV_SYSTEMS:
    print(f"\n=== dev / {system} ===")
    DEV_PREDS[system] = run_system(system, "dev", DEV_ITEMS, RANKINGS["dev"][system])

print("\ncompleted dev systems:", list(DEV_PREDS))



=== dev / hybrid ===
  [hybrid] 50/300  (224s, ETA 1120s)
  [hybrid] 100/300  (483s, ETA 965s)
  [hybrid] 150/300  (728s, ETA 728s)
  [hybrid] 200/300  (961s, ETA 481s)
  [hybrid] 250/300  (1206s, ETA 241s)
  [hybrid] 300/300  (1449s, ETA 0s)
Saved 300 predictions -> /kaggle/working/NLP_phase2/runs/dev/hybrid/predictions.jsonl

=== dev / hybrid_sentence ===
  [hybrid_sentence] 50/300  (240s, ETA 1198s)
  [hybrid_sentence] 100/300  (489s, ETA 978s)
  [hybrid_sentence] 150/300  (740s, ETA 740s)
  [hybrid_sentence] 200/300  (980s, ETA 490s)
  [hybrid_sentence] 250/300  (1226s, ETA 245s)
  [hybrid_sentence] 300/300  (1465s, ETA 0s)
Saved 300 predictions -> /kaggle/working/NLP_phase2/runs/dev/hybrid_sentence/predictions.jsonl

=== dev / hybrid_goldmask ===
  [hybrid_goldmask] 50/300  (219s, ETA 1094s)
  [hybrid_goldmask] 100/300  (462s, ETA 925s)
  [hybrid_goldmask] 150/300  (686s, ETA 686s)
  [hybrid_goldmask] 200/300  (923s, ETA 461s)
  [hybrid_goldmask] 250/300  (1150s, ETA 230s)
  [hyb

## §11 Scoring

Use the course `score.py` for dev metrics and official paired differences against Phase-1 dense. Within-type paired differences use the same `per_question` and `bootstrap` functions from that script.


In [13]:
# =============================================================================
# §11  Scoring with score.py and paired comparisons
# =============================================================================
# Scoring helpers adapt score.py output for Phase 2.
SCORES = {}          # system -> parsed score.py output
PAIRED = {}          # "A__vs__B" -> paired metrics
# SCORE_PATH was resolved in §1 (either in PACK_DIR or the dataset root).
assert SCORE_PATH.is_file(), f"score.py not found: {SCORE_PATH}"
print("using score.py:", SCORE_PATH)


def run_score(pred_path, split="dev", compare=None, partial=False, validate_only=False):
    return invoke_course_scorer(pred_path, SCORE_PATH, PACK_DIR, split=split,
                                  compare=compare, partial=partial,
                                  validate_only=validate_only)




def pred_path(system, split="dev"):
    return RUNS_DIR / split / system / "predictions.jsonl"


# When dev_limit is set, score.py must be invoked with --partial.
PARTIAL_DEV = CONFIG["dev_limit"] is not None

if CONFIG["run_dev"]:
    for system in DEV_SYSTEMS:
        print(f"\n{'=' * 20} {system} {'=' * 20}")
        text_out = run_score(pred_path(system), "dev", partial=PARTIAL_DEV)
        print(text_out)
        SCORES[system] = parse_score_output(text_out)
        (ARTIFACT_DIR / f"score_{system}.txt").write_text(text_out, encoding="utf-8")

    # ---- Official Phase-1 dense baseline (if attached) ----
    # The project asks for a comparison against the Phase-1 dense system. When a
    # Phase-1 predictions are read and saved in the required schema, then scored
    # with the same score.py on the matching questions.
    _src = PHASE1_ARTIFACTS["NLP_phase1_output/dev/dense/predictions.jsonl"]
    if _src:
        _dst_dir = RUNS_DIR / "dev" / "phase1_dense"
        _dst_dir.mkdir(parents=True, exist_ok=True)
        _rows = [r for r in read_jsonl(_src) if str(r["qid"]) in QDEV_BY_QID]
        if CONFIG["dev_limit"]:
            _allowed = {str(q["qid"]) for q in DEV_ITEMS}
            _rows = [r for r in _rows if str(r["qid"]) in _allowed]
        save_predictions(_rows, _dst_dir / "predictions.jsonl")
        _original_config = PHASE1_ARTIFACTS["NLP_phase1_output/dev/dense/run_config.yaml"]
        if _original_config is None:
            raise FileNotFoundError("Phase-1 dense run_config.yaml is required to preserve baseline provenance.")
        shutil.copy2(_original_config, _dst_dir / "run_config.yaml")
        print("\nPhase-1 dense baseline found:", _src)
        _txt = run_score(pred_path("phase1_dense"), "dev", partial=PARTIAL_DEV)
        print(_txt)
        SCORES["phase1_dense"] = parse_score_output(_txt)
        (ARTIFACT_DIR / "score_phase1_dense.txt").write_text(_txt, encoding="utf-8")
    else:
        raise FileNotFoundError("Phase-1 NLP_phase1_output/dev/dense/predictions.jsonl is required for the paired baseline comparison.")

    # ---- Paired comparisons ----
    comparisons = []
    if "hybrid" in DEV_SYSTEMS:
        for system in ("hybrid_sentence",):
            if system in DEV_SYSTEMS and (system, "hybrid") not in comparisons:
                comparisons.append((system, "hybrid"))
    if "phase1_dense" in SCORES and "hybrid" in DEV_SYSTEMS:
        comparisons.append(("hybrid", "phase1_dense"))

    for system, other in comparisons:
        print(f"\n{'=' * 12} paired: {system} minus {other} {'=' * 12}")
        text_out = run_score(pred_path(system), "dev",
                             compare=pred_path(other), partial=PARTIAL_DEV)
        print(text_out)
        PAIRED[f"{system}__vs__{other}"] = parse_score_output(text_out)["paired"]
        (ARTIFACT_DIR / f"paired_{system}_vs_{other}.txt").write_text(text_out, encoding="utf-8")

    # Paired within-type differences against the saved Phase-1 dense run.
    import importlib.util as _score_import
    _score_spec = _score_import.spec_from_file_location("course_score", SCORE_PATH)
    _course_score = _score_import.module_from_spec(_score_spec)
    _score_spec.loader.exec_module(_course_score)
    _a = {str(x["qid"]): x for x in read_jsonl(pred_path("hybrid"))}
    _b = {str(x["qid"]): x for x in read_jsonl(pred_path("phase1_dense"))}
    if set(_a) != set(_b):
        raise ValueError("Paired comparison requires identical hybrid and Phase-1 dense qids.")
    PAIRED_BY_TYPE = {}
    TYPE_RETRIEVAL_TRANSITIONS = {}
    for _qid in sorted(_a):
        _lab = LABELS_BY_QID[_qid]
        _type = _lab["type"]
        _ma, _mb = _course_score.per_question(_a[_qid], _lab), _course_score.per_question(_b[_qid], _lab)
        _transitions = TYPE_RETRIEVAL_TRANSITIONS.setdefault(_type, {"hybrid_only": 0, "phase1_only": 0, "both": 0, "neither": 0})
        _ha, _hb = bool(_ma["hitall@10"]), bool(_mb["hitall@10"])
        _transition = "both" if _ha and _hb else "hybrid_only" if _ha else "phase1_only" if _hb else "neither"
        _transitions[_transition] += 1
        _bucket = PAIRED_BY_TYPE.setdefault(_type, {})
        for _metric in _ma.keys() & _mb.keys():
            _bucket.setdefault(_metric, []).append(_ma[_metric] - _mb[_metric])
    PAIRED_BY_TYPE = {
        t: {m: {"value": 100 * sum(v) / len(v),
                 "ci": tuple(100 * x for x in _course_score.bootstrap(v)), "n": len(v)}
            for m, v in metrics.items()}
        for t, metrics in PAIRED_BY_TYPE.items()
    }
    (ARTIFACT_DIR / "paired_by_type.json").write_text(
        json.dumps(PAIRED_BY_TYPE, ensure_ascii=False, indent=2), encoding="utf-8")

    (ARTIFACT_DIR / "type_retrieval_transitions.json").write_text(
        json.dumps(TYPE_RETRIEVAL_TRANSITIONS, ensure_ascii=False, indent=2), encoding="utf-8")

    # ---- Summary table ----
    import pandas as pd

    rows = []
    for system in SCORES:
        ov = SCORES[system]["overall"]
        rows.append({
            "system": system,
            "n": SCORES[system]["overall_n"],
            **{k: round(ov[k]["value"], 1) for k in
               ("recall@10", "hitall@10", "mrr", "ndcg@10", "em", "f1",
                "cite_precision", "cite_recall", "abstained") if k in ov},
        })
    SUMMARY_DEV = pd.DataFrame(rows).set_index("system")
    print("\n=== DEV SUMMARY (percent, from score.py) ===")
    display(SUMMARY_DEV)

    summary_export = {
        "scores": {
            system: {
                "overall": {m: v["value"] for m, v in s["overall"].items()},
                "by_type": {t: {m: v["value"] for m, v in s["by_type"][t].items()}
                            for t in s["by_type"]},
                "type_n": s["type_n"],
                "n": s["overall_n"],
            }
            for system, s in SCORES.items()
        },
        "paired": {k: {m: v["value"] for m, v in p.items()} for k, p in PAIRED.items()},
    }
    (ARTIFACT_DIR / "summary_dev.json").write_text(
        json.dumps(summary_export, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved:", ARTIFACT_DIR / "summary_dev.json")


using score.py: /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/course_pack_v1/course_pack_v1/score.py

==================== hybrid ====================
valid: /kaggle/working/NLP_phase2/runs/dev/hybrid/predictions.jsonl (300 predictions)

overall (n=300)
  recall@2          59.2   [ 56.4,  62.2]
  hitall@2          23.3   [ 18.7,  28.3]
  recall@5          69.3   [ 66.5,  72.2]
  hitall@5          37.7   [ 32.0,  43.3]
  recall@10         73.2   [ 70.2,  75.8]
  hitall@10         44.3   [ 38.7,  50.0]
  mrr               95.2   [ 93.3,  97.1]
  ndcg@10           74.7   [ 72.3,  76.7]
  em                27.7   [ 22.7,  33.0]
  f1                36.7   [ 31.7,  41.6]
  cite_precision    50.9   [ 44.9,  56.4]
  cite_recall       34.6   [ 30.4,  38.9]
  abstained         36.3   [ 31.0,  42.0]

type: bridge_comparison (n=66)
  recall@2          45.8   [ 43.6,  48.1]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5          55.3   [ 53.0,  58.0]
  hitall@5           1.5   [

,n,recall@10,hitall@10,mrr,ndcg@10,em,f1,cite_precision,cite_recall,abstained
system,,,,,,,,,,
hybrid,300,73.2,44.3,95.2,74.7,27.7,36.7,50.9,34.6,36.3
hybrid_sentence,300,73.3,44.0,95.6,74.9,28.3,38.3,49.5,34.1,37.0
hybrid_goldmask,300,0.0,0.0,0.0,0.0,0.7,2.4,0.0,0.0,84.0
phase1_dense,300,79.2,55.3,98.8,80.6,37.0,46.5,52.7,36.0,35.0


saved: /kaggle/working/NLP_phase2/artifacts/summary_dev.json


## §12 Gold-mask abstention

Measure masked abstention, correct and wrong answers separately, and normal-run over-abstention when all gold IDs are in the packed context. Removing gold IDs does not prove that alternative evidence is absent.


In [14]:
# =============================================================================
# §12  Gold-mask abstention analysis: grounded vs parametric vs hallucination vs over-abstention
# =============================================================================

def normalize_answer(s):
    s = "".join(ch for ch in str(s).lower() if ch not in set(string.punctuation))
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", s).split())


def is_correct(pred, label):
    if pred.get("abstained"):
        return False
    golds = [label["answer"]] + list(label.get("aliases", []))
    return any(normalize_answer(pred["answer"]) == normalize_answer(g) for g in golds)


ABSTENTION_ROWS = []
if CONFIG["run_dev"] and "hybrid" in DEV_PREDS and "hybrid_goldmask" in DEV_PREDS:
    normal = {r["qid"]: r for r in DEV_PREDS["hybrid"]}
    masked = {r["qid"]: r for r in DEV_PREDS["hybrid_goldmask"]}

    for item in DEV_ITEMS:
        qid = str(item["qid"])
        label = LABELS_BY_QID[qid]
        gold = set(gold_ids_of(qid))
        ctx_normal = set(CONTEXT_USED["hybrid"].get(qid, []))

        p_n, p_m = normal[qid], masked[qid]
        n_ok, m_ok = is_correct(p_n, label), is_correct(p_m, label)

        # Mutually exclusive observed outcomes, not proof of evidence absence.
        if p_m["abstained"]:
            cat = "masked_abstained"
        elif m_ok:
            cat = "masked_correct"
        else:
            cat = "masked_wrong"

        ABSTENTION_ROWS.append({
            "qid": qid,
            "type": item.get("type", "unknown"),
            "gold_in_context": float(gold <= ctx_normal),
            "normal_abstained": float(p_n["abstained"]),
            "normal_correct": float(n_ok),
            "normal_over_abstained": float(bool(p_n["abstained"]) and gold <= ctx_normal),
            "masked_abstained": float(p_m["abstained"]),
            "masked_correct": float(m_ok),
            "category": cat,
        })

    import pandas as pd
    ABST_DF = pd.DataFrame(ABSTENTION_ROWS)
    ABST_DF.to_csv(ARTIFACT_DIR / "abstention_analysis.csv", index=False)

    dist = ABST_DF["category"].value_counts()
    total = len(ABST_DF)
    print("=== gold-mask abstention experiment (dev) ===")
    for cat, n in dist.items():
        print(f"  {cat:22s} {n:4d}  ({n / total:5.1%})")
    print(f"\n  normal abstention rate : {ABST_DF['normal_abstained'].mean():.1%}")
    print(f"  masked abstention rate : {ABST_DF['masked_abstained'].mean():.1%}")
    print(f"  masked wrong answers   : {(ABST_DF.category == 'masked_wrong').sum()} "
          f"({(ABST_DF.category == 'masked_wrong').mean():.1%})")
    print(f"  masked correct (evidence may exist): {(ABST_DF.category == 'masked_correct').sum()} "
          f"({(ABST_DF.category == 'masked_correct').mean():.1%})")

    ABST_SUMMARY = {
        "category_counts": dist.to_dict(),
        "normal_abstention_rate": float(ABST_DF["normal_abstained"].mean()),
        "masked_abstention_rate": float(ABST_DF["masked_abstained"].mean()),
        "masked_wrong_rate": float((ABST_DF.category == "masked_wrong").mean()),
        "masked_correct_rate": float((ABST_DF.category == "masked_correct").mean()),
        "over_abstention_rate": float(ABST_DF["normal_over_abstained"].mean()),
        "n": total,
    }
    (ARTIFACT_DIR / "abstention_summary.json").write_text(
        json.dumps(ABST_SUMMARY, ensure_ascii=False, indent=2), encoding="utf-8")

    print("\nPer-question-type abstention rates (gold-masked run):")
    print(ABST_DF.groupby("type")[["masked_abstained", "masked_correct"]].mean().round(3))
else:
    ABST_DF, ABST_SUMMARY = None, {}
    print("gold-mask experiment skipped (run_goldmask=False or no dev run)")


=== gold-mask abstention experiment (dev) ===
  masked_abstained        252  (84.0%)
  masked_wrong             46  (15.3%)
  masked_correct            2  ( 0.7%)

  normal abstention rate : 36.3%
  masked abstention rate : 84.0%
  masked wrong answers   : 46 (15.3%)
  masked correct (evidence may exist): 2 (0.7%)

Per-question-type abstention rates (gold-masked run):
                   masked_abstained  masked_correct
type                                               
bridge_comparison             0.924           0.000
comparison                    0.863           0.000
compositional                 0.848           0.016
inference                     0.611           0.000


## §13 Additional analyses

Results by type, bridge-comparison metrics and a descriptive error taxonomy are calculated from this run. Examples and supporting IDs are saved to `error_analysis.txt`.


In [15]:
# =============================================================================
# §13  Per-type analysis, bridge bottleneck, and error taxonomy
# =============================================================================
import pandas as pd

TYPE_ORDER = ["comparison", "bridge_comparison", "compositional", "inference"]

if CONFIG["run_dev"]:
    # ---- per-type metrics table ----
    rows = []
    for system in DEV_SYSTEMS:
        for qtype in TYPE_ORDER:
            if qtype not in SCORES[system]["by_type"]:
                continue
            m = SCORES[system]["by_type"][qtype]
            rows.append({
                "system": system, "type": qtype, "n": SCORES[system]["type_n"].get(qtype, 0),
                **{k.replace("@", "_"): round(m[k]["value"], 1) for k in
                   ("recall@10", "hitall@10", "mrr", "ndcg@10", "em", "f1",
                    "cite_precision", "cite_recall", "abstained") if k in m},
            })
    BY_TYPE_DF = pd.DataFrame(rows)
    print("=== metrics by question type ===")
    display(BY_TYPE_DF.round(1))

    # ---- bridge bottleneck ----
    print("\n=== bridge_comparison bottleneck ===")
    bridge_rows = []
    for system in DEV_SYSTEMS:
        bt = SCORES[system]["by_type"].get("bridge_comparison")
        if not bt:
            continue
        bridge_rows.append({
            "system": system,
            "n": SCORES[system]["type_n"]["bridge_comparison"],
            "recall@2": bt["recall@2"]["value"], "recall@5": bt["recall@5"]["value"],
            "recall@10": bt["recall@10"]["value"],
            "hitall@2": bt["hitall@2"]["value"], "hitall@5": bt["hitall@5"]["value"],
            "hitall@10": bt["hitall@10"]["value"],
            "mrr": bt["mrr"]["value"], "em": bt["em"]["value"],
        })
    BRIDGE_DF = pd.DataFrame(bridge_rows).set_index("system").round(1)
    display(BRIDGE_DF)
    avg_gold_bridge = np.mean([len(gold_ids_of(str(q["qid"]))) for q in DEV_ITEMS
                               if q.get("type") == "bridge_comparison"])
    print(f"average gold documents per bridge question: {avg_gold_bridge:.2f}")

    # ---- error taxonomy ----
    ERR = []
    hybrid = {r["qid"]: r for r in DEV_PREDS["hybrid"]}
    for item in DEV_ITEMS:
        qid = str(item["qid"])
        label = LABELS_BY_QID[qid]
        gold = set(gold_ids_of(qid))
        ctx = set(CONTEXT_USED["hybrid"].get(qid, []))
        rec = RANKINGS["dev"]["hybrid"][qid]
        pred = hybrid[qid]
        ok = is_correct(pred, label)

        if not (gold <= set(rec[:10])):
            err = "retrieval_failure"
        elif not (gold <= ctx):
            err = "packing_evidence_gap"
        elif not ok and pred["abstained"]:
            err = "over_abstention"
        elif not ok:
            err = "generation_error"
        elif ok and not (gold <= set(pred["cited"])):
            err = "citation_gap"
        else:
            err = "ok"
        ERR.append({"qid": qid, "type": item.get("type"), "error": err,
                    "gold_in_context": float(gold <= ctx),
                    "abstained": float(pred["abstained"]),
                    "answered": float(not pred["abstained"]),
                    "correct": float(ok),
                    "n_cited": len(pred["cited"]),
                    "n_gold": len(gold)})
    ERR_DF = pd.DataFrame(ERR)
    ERR_DF.to_csv(ARTIFACT_DIR / "error_taxonomy.csv", index=False)

    print("\n=== hybrid error taxonomy (dev) ===")
    counts = ERR_DF["error"].value_counts()
    for k, v in counts.items():
        print(f"  {k:20s} {v:4d}  ({v / len(ERR_DF):5.1%})")

    # ---- write error examples ----
    def fmt_example(row, bucket):
        qid = row["qid"]
        item = QDEV_BY_QID[qid]
        label = LABELS_BY_QID[qid]
        pred = hybrid[qid]
        packed = CONTEXT_USED["hybrid"].get(qid, [])
        lines = [
            f"QID: {qid} | type: {item.get('type')} | bucket: {bucket}",
            f"question        : {item['question']}",
            f"gold answer     : {label['answer']} | aliases: {label.get('aliases', [])}",
            f"gold doc ids    : {gold_ids_of(qid)}",
            f"retrieved top10 : {RANKINGS['dev']['hybrid'][qid]}",
            f"context (packed): {packed}",
            f"system answer   : {pred['answer']!r} | abstained={pred['abstained']} | cited={pred['cited']}",
            f"raw output      : {RAW_OUTPUTS['hybrid'].get(qid, '')[:400]!r}",
        ]
        return "\n".join(lines)

    ERROR_TXT = OUTPUT_DIR / "error_analysis.txt"
    with ERROR_TXT.open("w", encoding="utf-8") as fh:
        fh.write("HYBRID RAG — ERROR ANALYSIS (dev)\n" + "=" * 60 + "\n\n")
        for bucket in ("retrieval_failure", "packing_evidence_gap", "over_abstention", "generation_error", "citation_gap"):
            subset = ERR_DF[ERR_DF.error == bucket].head(5)
            n_total = int((ERR_DF.error == bucket).sum())
            fh.write(f"\n### {bucket}  ({n_total} cases, {n_total/len(ERR_DF):.1%} of dev)\n\n")
            for _, row in subset.iterrows():
                fh.write(fmt_example(row, bucket) + "\n\n" + "-" * 50 + "\n\n")
    print("saved:", ERROR_TXT)

    print("\n=== sample retrieval failures ===")
    for _, row in ERR_DF[ERR_DF.error == "retrieval_failure"].head(3).iterrows():
        print(fmt_example(row, "retrieval_failure") + "\n")


=== metrics by question type ===


,system,type,n,recall_10,hitall_10,mrr,ndcg_10,em,f1,cite_precision,cite_recall,abstained
0,hybrid,comparison,73,100.0,100.0,97.3,96.3,71.2,85.1,71.9,70.5,0.0
1,hybrid,bridge_comparison,66,59.1,7.6,99.2,68.3,18.2,22.7,54.8,25.4,39.4
2,hybrid,compositional,125,64.4,28.8,94.5,66.8,12.8,16.6,31.6,17.2,62.4
3,hybrid,inference,36,75.0,52.8,86.2,69.7,8.3,34.2,68.1,38.9,13.9
4,hybrid_sentence,comparison,73,100.0,100.0,97.9,96.5,67.1,85.8,71.9,69.2,0.0
5,hybrid_sentence,bridge_comparison,66,59.8,6.1,98.5,68.6,19.7,25.5,47.7,21.6,45.5
6,hybrid_sentence,compositional,125,64.4,28.8,95.2,66.9,16.0,20.5,35.2,18.8,58.4
7,hybrid_sentence,inference,36,75.0,52.8,86.7,70.0,8.3,27.4,56.9,38.9,22.2
8,hybrid_goldmask,comparison,73,0.0,0.0,0.0,0.0,0.0,2.7,0.0,0.0,86.3
9,hybrid_goldmask,bridge_comparison,66,0.0,0.0,0.0,0.0,0.0,1.5,0.0,0.0,92.4



=== bridge_comparison bottleneck ===


,n,recall@2,recall@5,recall@10,hitall@2,hitall@5,hitall@10,mrr,em
system,,,,,,,,,
hybrid,66,45.8,55.3,59.1,0.0,1.5,7.6,99.2,18.2
hybrid_sentence,66,45.8,56.1,59.8,0.0,1.5,6.1,98.5,19.7
hybrid_goldmask,66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


average gold documents per bridge question: 4.00

=== hybrid error taxonomy (dev) ===
  retrieval_failure     167  (55.7%)
  generation_error       56  (18.7%)
  ok                     37  (12.3%)
  citation_gap           29  ( 9.7%)
  over_abstention         8  ( 2.7%)
  packing_evidence_gap    3  ( 1.0%)
saved: /kaggle/working/NLP_phase2/error_analysis.txt

=== sample retrieval failures ===
QID: q0002 | type: bridge_comparison | bucket: retrieval_failure
question        : Which film has the director who was born later, The Third Side Of The River or Escape From L.A.?
gold answer     : The Third Side Of The River | aliases: ['The Third Side of the River', 'Third Side of the River']
gold doc ids    : ['p03489', 'p01066', 'p00620', 'p01750']
retrieved top10 : ['p03489', 'p01066', 'p01068', 'p03449', 'p00835', 'p02156', 'p03756', 'p00788', 'p01883', 'p03223']
context (packed): ['p03489', 'p01066', 'p01068', 'p03449', 'p00835', 'p02156', 'p03756', 'p00788', 'p01883', 'p03223']
system answ

## §14 Hidden split

Generate hybrid predictions and validate their structure with `score.py --validate-only`. No hidden labels or hidden accuracy are assumed.


In [16]:
# =============================================================================
# §14  Run final systems on the hidden split and validate outputs
# =============================================================================
HIDDEN_PREDS = {}
HIDDEN_SYSTEMS = [s for s in CONFIG["hidden_systems"] if s in RANKINGS["hidden"]]
assert HIDDEN_SYSTEMS, "No systems selected for the hidden split (check CONFIG['hidden_systems'])"

for system in HIDDEN_SYSTEMS:
    print(f"\n=== hidden / {system} ===")
    HIDDEN_PREDS[system] = run_system(system, "hidden", questions_hidden,
                                      RANKINGS["hidden"][system])

for system in HIDDEN_SYSTEMS:
    path = pred_path(system, "hidden")
    print(f"\nvalidating {path}")
    print(run_score(path, "hidden", validate_only=True))
    rows = read_jsonl(path)
    assert len(rows) == len(questions_hidden), (
        f"expected {len(questions_hidden)} hidden predictions, got {len(rows)}"
    )
    assert {r["qid"] for r in rows} == {str(q["qid"]) for q in questions_hidden}, (
        "hidden qid set does not match questions_hidden.jsonl"
    )
    assert set(rows[0]) == REQUIRED_FIELDS, (
        f"hidden predictions must contain exactly fields {REQUIRED_FIELDS}, got {set(rows[0])}"
    )
print("hidden predictions validated.")



=== hidden / hybrid ===
  [hybrid] 50/300  (230s, ETA 1152s)
  [hybrid] 100/300  (488s, ETA 975s)
  [hybrid] 150/300  (722s, ETA 722s)
  [hybrid] 200/300  (965s, ETA 482s)
  [hybrid] 250/300  (1194s, ETA 239s)
  [hybrid] 300/300  (1439s, ETA 0s)
Saved 300 predictions -> /kaggle/working/NLP_phase2/runs/hidden/hybrid/predictions.jsonl

validating /kaggle/working/NLP_phase2/runs/hidden/hybrid/predictions.jsonl
valid: /kaggle/working/NLP_phase2/runs/hidden/hybrid/predictions.jsonl (300 predictions)

hidden predictions validated.


## §15.1 Figures

Figures use this run's score and gold-mask analysis outputs. Missing metrics are not treated as zero.


In [17]:
# =============================================================================
# §15-1  Figures
# =============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

FIGS = {}

if CONFIG["run_dev"] and SCORES:
    show_systems = [s for s in ("phase1_dense", "dense", "hybrid",
                                "hybrid_sentence") if s in SCORES]
    metrics = [("em", "EM"), ("f1", "F1"), ("hitall@10", "HitAll@10"),
               ("recall@10", "Recall@10"), ("cite_precision", "Cite-P")]
    metrics = [(m, lab) for m, lab in metrics if all(m in SCORES[s]["overall"] for s in show_systems)]

    # ---- Figure 1: overall comparison across key metrics ----
    fig, ax = plt.subplots(figsize=(7.4, 3.2), constrained_layout=True)
    x = np.arange(len(metrics)); w = 0.8 / max(len(show_systems), 1)
    for i, system in enumerate(show_systems):
        vals = [SCORES[system]["overall"][m]["value"] for m, _ in metrics]
        bars = ax.bar(x + (i - (len(show_systems) - 1) / 2) * w, vals, w, label=system)
        ax.bar_label(bars, fmt="%.0f", fontsize=6, padding=1)
    ax.set_xticks(x); ax.set_xticklabels([lab for _, lab in metrics], fontsize=8)
    ax.set_ylim(0, 105); ax.set_ylabel("score (%)", fontsize=8)
    ax.legend(fontsize=7, ncol=3); ax.tick_params(labelsize=7)
    ax.set_title("Dev overall: retrieval variants (score.py)", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    FIGS["overall"] = FIG_DIR / "fig_overall.png"
    fig.savefig(FIGS["overall"], dpi=200); plt.close(fig)

    # ---- Figure 2: per-question-type (EM and HitAll@10) ----
    types = [t for t in TYPE_ORDER if t in SCORES["hybrid"]["by_type"]]
    if types:
        fig, axes = plt.subplots(1, 2, figsize=(7.4, 2.9), constrained_layout=True)
        for ax, metric, title in ((axes[0], "hitall@10", "HitAll@10 by type"),
                                  (axes[1], "em", "EM by type")):
            x = np.arange(len(types)); w = 0.8 / max(len(show_systems), 1)
            for i, system in enumerate(show_systems):
                vals = [SCORES[system]["by_type"].get(t, {}).get(metric, {}).get("value", np.nan)
                        for t in types]
                bars = ax.bar(x + (i - (len(show_systems) - 1) / 2) * w, vals, w,
                              label=system)
                ax.bar_label(bars, fmt="%.0f", fontsize=5, padding=1)
            ax.set_xticks(x)
            ax.set_xticklabels([t.replace("_", "\n") for t in types], fontsize=6.5)
            ax.set_ylim(0, 105); ax.set_title(title, fontsize=9)
            ax.tick_params(labelsize=7); ax.spines[["top", "right"]].set_visible(False)
        axes[0].legend(fontsize=6, ncol=2)
        FIGS["by_type"] = FIG_DIR / "fig_by_type.png"
        fig.savefig(FIGS["by_type"], dpi=200); plt.close(fig)

    # ---- Figure 3: abstention experiment ----
    if ABST_SUMMARY.get("category_counts"):
        cats = ["masked_abstained", "masked_correct", "masked_wrong"]
        vals = [ABST_SUMMARY["category_counts"].get(c, 0) for c in cats]
        order = np.argsort(vals)[::-1]
        fig, ax = plt.subplots(figsize=(6.4, 2.6), constrained_layout=True)
        bars = ax.bar([cats[i] for i in order], [vals[i] for i in order], color="#4C72B0")
        ax.bar_label(bars, fontsize=7)
        ax.set_title("Gold-masked abstention experiment (dev, n=%d)" % ABST_SUMMARY["n"], fontsize=9)
        ax.tick_params(labelsize=7, axis="x", rotation=18)
        ax.spines[["top", "right"]].set_visible(False)
        FIGS["abstention"] = FIG_DIR / "fig_abstention.png"
        fig.savefig(FIGS["abstention"], dpi=200); plt.close(fig)

print("figures:", {k: str(v) for k, v in FIGS.items()})


figures: {'overall': '/kaggle/working/NLP_phase2/figures/fig_overall.png', 'by_type': '/kaggle/working/NLP_phase2/figures/fig_by_type.png', 'abstention': '/kaggle/working/NLP_phase2/figures/fig_abstention.png'}


In [18]:
# =============================================================================
# §15-2  RTL HTML reports, PDF rendering and submission archive
# =============================================================================
import base64
import html
import urllib.request
from weasyprint import HTML

if not CONFIG["run_dev"] or not {"phase1_dense", "hybrid", "hybrid_sentence", "hybrid_goldmask"} <= set(SCORES):
    raise RuntimeError("A full dev evaluation with all required systems is needed for the report.")
if CONFIG["dev_limit"] is not None or SCORES["hybrid"]["overall_n"] != len(questions_dev):
    raise RuntimeError("A full dev run is required for the report.")
if not {"hybrid", "hybrid_sentence", "hybrid_goldmask"} <= set(DEV_PREDS):
    raise RuntimeError("Missing required dev predictions.")

# Download Vazirmatn once; HTML embeds the font and figures as data URIs.
font_dir = OUTPUT_DIR / "fonts"
font_dir.mkdir(parents=True, exist_ok=True)
font_path = font_dir / "Vazirmatn-Regular.ttf"
if not font_path.exists():
    font_url = "https://cdn.jsdelivr.net/gh/rastikerdar/vazirmatn@v33.003/fonts/ttf/Vazirmatn-Regular.ttf"
    try:
        urllib.request.urlretrieve(font_url, font_path)
    except Exception as exc:
        raise RuntimeError("Cannot download Vazirmatn; enable Kaggle internet or place the font in OUTPUT_DIR/fonts.") from exc
font_b64 = base64.b64encode(font_path.read_bytes()).decode("ascii")

def esc(value):
    return html.escape(str(value), quote=True)

def ltr(value):
    return '<bdi dir="ltr" class="latin">' + esc(value) + '</bdi>'

def metric(system, name):
    v = SCORES.get(system, {}).get("overall", {}).get(name)
    return "—" if v is None else f"{v['value']:.1f}"

def pmetric(pair, name):
    v = PAIRED.get(pair, {}).get(name)
    return "—" if v is None else f"{v['value']:+.1f} [{v['ci'][0]:+.1f}, {v['ci'][1]:+.1f}]"

def type_delta(qtype, name):
    v = PAIRED_BY_TYPE.get(qtype, {}).get(name)
    return "—" if v is None else f"{v['value']:+.1f} [{v['ci'][0]:+.1f}, {v['ci'][1]:+.1f}]"

def table(headers, rows, css_class=""):
    head = ''.join(f'<th>{esc(h)}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{c}</td>' for c in row) + '</tr>' for row in rows)
    return f'<table class="{esc(css_class)}"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'

def figure(name, alt):
    path = FIGS.get(name)
    if not path or not Path(path).is_file():
        return ''
    data = base64.b64encode(Path(path).read_bytes()).decode('ascii')
    return f'<figure><img src="data:image/png;base64,{data}" alt="{esc(alt)}"><figcaption>{esc(alt)}</figcaption></figure>'

systems = ["phase1_dense", "hybrid", "hybrid_sentence", "hybrid_goldmask"]
metrics = ("recall@10", "hitall@10", "mrr", "ndcg@10", "em", "f1", "cite_precision", "cite_recall", "abstained")
for system in systems:
    if SCORES[system]["overall_n"] != len(questions_dev):
        raise RuntimeError(f"Mismatched dev count for {system}.")
rows = [[ltr(system)] + [ltr(metric(system, m)) for m in metrics] for system in systems]
overall_table = table(["سامانه", "R@10", "H@10", "MRR", "nDCG", "EM", "F1", "Cite-P", "Cite-R", "Abst"], rows, 'metrics')
key = "hybrid__vs__phase1_dense"
if key not in PAIRED:
    raise RuntimeError("Missing paired hybrid vs Phase-1 dense comparison.")
pair_table = table(["معیار", "hybrid − phase1_dense؛ بازه ۹۵٪"],
    [[ltr(m), ltr(pmetric(key, m))] for m in ("hitall@10", "em", "f1", "cite_precision", "cite_recall")])
ptype_rows = [[ltr(t), ltr(next(iter(PAIRED_BY_TYPE[t].values()))["n"]),
               ltr(type_delta(t, "hitall@10")), ltr(type_delta(t, "em"))]
              for t in sorted(PAIRED_BY_TYPE) if PAIRED_BY_TYPE[t]]
ptype_table = table(["نوع سؤال", "تعداد", "تغییر HitAll@10", "تغییر EM"], ptype_rows)
_type_hits = [(t, data['hitall@10']['value']) for t, data in PAIRED_BY_TYPE.items() if 'hitall@10' in data]
_type_note = ''
if _type_hits:
    _best = max(_type_hits, key=lambda x: x[1])
    _worst = min(_type_hits, key=lambda x: x[1])
    _type_note = (f"در این اجرا بیشترین تفاضل مشاهده‌شدهٔ HitAll@10 در نوع {ltr(_best[0])} "
                  f"با {ltr(f'{_best[1]:+.1f}')} و کمترین آن در نوع {ltr(_worst[0])} "
                  f"با {ltr(f'{_worst[1]:+.1f}')} واحد درصد است. این توصیف علت تفاوت را ثابت نمی‌کند؛ "
                  "تعداد سؤال و فاصلهٔ اطمینان هر ردیف نیز باید دیده شوند.")
transition_rows = [[ltr(t), ltr(v["hybrid_only"]), ltr(v["phase1_only"]), ltr(v["both"]), ltr(v["neither"])]
                   for t, v in sorted(TYPE_RETRIEVAL_TRANSITIONS.items())]
transition_table = table(["نوع", "فقط hybrid موفق", "فقط فاز اول موفق", "هر دو موفق", "هیچ‌کدام موفق"], transition_rows)
if _type_hits:
    _best = max(_type_hits, key=lambda x: x[1])
    _transition = TYPE_RETRIEVAL_TRANSITIONS[_best[0]]
    _positive_note = ("بیشترین افزایش مشاهده‌شده" if _best[1] > 0 else "بیشترین اختلاف مشاهده‌شده (بهبود مثبت ثبت نشده است)")
    _type_why = (f"{_positive_note} برای {ltr(_best[0])} برابر {ltr(f'{_best[1]:+.1f}')} واحد درصد است؛ "
                 f"در این نوع {ltr(_transition['hybrid_only'])} پرسش فقط در hybrid و "
                 f"{ltr(_transition['phase1_only'])} پرسش فقط در فاز اول همهٔ اسناد طلایی را دارند. "
                 "تفاوت این دو تعداد تغییر HitAll را توضیح می‌دهد، اما علت عملکرد بازیاب را ثابت نمی‌کند. "
                 "فرضیهٔ قابل بررسی این است که BM25 در برخی پرسش‌ها واژه‌های خاص موجود در اسناد را بهتر بازیابی کند؛ "
                 "برای تأیید باید فهرست‌های هر دو بازیاب و شواهد هر سؤال دستی بررسی شوند.")
else:
    _type_why = "دادهٔ کافی برای تفکیک نوع در دسترس نیست."

mask_table = table(["نتیجه", "درصد از همهٔ پرسش‌ها"], [
    ["امتناع در حالت ماسک", ltr(f"{100*ABST_SUMMARY['masked_abstention_rate']:.1f}%")],
    ["پاسخ درست در حالت ماسک", ltr(f"{100*ABST_SUMMARY['masked_correct_rate']:.1f}%")],
    ["پاسخ نادرست در حالت ماسک", ltr(f"{100*ABST_SUMMARY['masked_wrong_rate']:.1f}%")],
    ["امتناع با وجود حضور همهٔ اسناد طلایی در اجرای عادی", ltr(f"{100*ABST_SUMMARY['over_abstention_rate']:.1f}%")],
])
counts = ERR_DF["error"].value_counts().to_dict()
error_labels = {"retrieval_failure":"دست‌کم یک سند طلایی در ده سند بازیابی‌شده نبود",
                "packing_evidence_gap":"همهٔ اسناد طلایی بازیابی شدند ولی همگی در زمینه جا نگرفتند",
                "over_abstention":"امتناع با وجود حضور همهٔ اسناد طلایی در زمینه",
                "generation_error":"پاسخ نادرست با حضور همهٔ اسناد طلایی در زمینه",
                "citation_gap":"پاسخ درست با ارجاع ناکامل به اسناد طلایی", "ok":"بدون خطا در این دسته‌بندی"}
error_rows = [[esc(error_labels.get(name,name)), ltr(counts.get(name, 0)), ltr(f"{100*counts.get(name,0)/len(ERR_DF):.1f}%")]
              for name in ("retrieval_failure","packing_evidence_gap","over_abstention","generation_error","citation_gap","ok")]
error_table = table(["دسته‌بندی ترتیبی خطا", "تعداد", "درصد از dev"], error_rows)

header = f'''<div class="bism">بسم تعالی</div>
<header><div class="eyebrow">پروژهٔ پردازش زبان طبیعی پیشرفته · فاز دوم</div>
<h1>بازیابی ترکیبی و پاسخ مستند</h1><p class="subtitle">گزارش فنی و ارزیابی سامانه</p></header>
<div class="identity"><div><span>تهیه‌کننده</span><strong>{esc(STUDENT_NAME)}</strong></div>
<div><span>شمارهٔ دانشجویی</span><strong>{ltr(STD_ID)}</strong></div></div>
{table(['مشخصات اجرا', 'مقدار'], [
 ['مخزن پروژه', ltr(GITHUB_URL)], ['شناسهٔ commit اعلام‌شده', ltr(GITHUB_COMMIT)],
 ['مدل embedding', ltr(CONFIG['embedding_model'])], ['مدل تولید پاسخ', ltr(CONFIG['llm_model'])],
 ['سخت‌افزار اعلام‌شده', 'دو GPU از نوع '+ltr('Tesla T4')+' در '+ltr('Kaggle')+'، با دقت '+ltr('float16')],
 ['تاریخ اعلام‌شده (شمسی)', ltr(REPORT_DATE)]], 'meta')}'''

dense_h = float(SCORES['phase1_dense']['overall']['hitall@10']['value'])
hybrid_h = float(SCORES['hybrid']['overall']['hitall@10']['value'])
hybrid_em = float(SCORES['hybrid']['overall']['em']['value'])
dense_em = float(SCORES['phase1_dense']['overall']['em']['value'])
pair_em = PAIRED[key].get('em')
ci_note = ('بازهٔ اطمینان اختلاف پاسخ دقیق شامل صفر است؛ داده‌های این اجرا جهت قطعی تغییر را روشن نمی‌کنند.'
           if pair_em and pair_em['ci'][0] <= 0 <= pair_em['ci'][1]
           else 'برای تفسیر اختلاف پاسخ دقیق، اندازهٔ اثر و بازهٔ اطمینان جدول را با هم بررسی کنید.')
rank_audit = DENSE_RANK_AUDIT['exact_order_percent']
result_text = (f"در این اجرا مقدار {ltr('HitAll@10')} برای dense ذخیره‌شدهٔ فاز اول {ltr(f'{dense_h:.1f}%')} "
               f"و برای hybrid {ltr(f'{hybrid_h:.1f}%')} است؛ پاسخ دقیق به‌ترتیب {ltr(f'{dense_em:.1f}%')} "
               f"و {ltr(f'{hybrid_em:.1f}%')} است. {ci_note}")
methods = (f"ارزیابی بر {ltr(len(questions_dev))} پرسش dev و {ltr(len(DOC_IDS))} سند انجام شد. "
           f"بازیاب متراکم فاز اول و {ltr('BM25')} دست‌نویس با {ltr('k1=1.5, b=0.75')} کار می‌کنند؛ "
           f"خروجی آن‌ها با {ltr('RRF')} و ثابت {ltr(CONFIG['rrf_k'])} ادغام می‌شود. "
           f"تا {ltr(CONFIG['k_context'])} سند به‌ترتیب رتبه و تا سقف {ltr(CONFIG['max_context_tokens'])} توکن "
           f"tokenizer مولد در زمینه قرار می‌گیرد. بازرتبه‌بند در این نسخه استفاده نشده است.")
comparison = (f"مقایسهٔ اصلی hybrid منهای خروجی ذخیره‌شدهٔ dense فاز اول روی پرسش‌های یکسان است. "
              f"رتبه‌بندی ده سند dense بازسازی‌شده در {ltr(f'{rank_audit:.1f}%')} پرسش‌ها با خروجی فاز اول "
              f"هم‌ترتیب بود. پاسخ‌های تاریخی فاز اول با تنظیمات تولید و زمینهٔ متفاوت ساخته شده‌اند؛ "
              f"بنابراین اختلاف EM و F1 اثر خالص بازیاب نیست.")
mask_text = ("در gold-mask اسناد طلایی پیش از top-k از هر دو بازیاب حذف می‌شوند؛ صفر بودن معیارهای مبتنی بر gold "
             "در این آزمایش قابل انتظار است. پاسخ درست بدون gold لزوماً دانش پارامتری و پاسخ نادرست لزوماً "
             "اثبات توهم نیست، زیرا ممکن است سند دیگری شواهد داشته باشد. ردیف امتناع عادی جدا از سه دستهٔ ماسک است.")
error_text = (f"در دسته‌بندی ترتیبی hybrid، {ltr(counts.get('retrieval_failure',0))} مورد در مرحلهٔ بازیابی و "
              f"{ltr(counts.get('packing_evidence_gap',0))} مورد در مرحلهٔ packing در نخستین شرط خطا قرار گرفتند. "
              "این دسته‌ها علل قطعی نیستند: یک سؤال ممکن است چند مشکل هم‌زمان داشته باشد و فقط نخستین شرط ثبت می‌شود. "
              "استناد از نظر تطبیق شناسه با gold سنجیده می‌شود، نه صحت معنایی ادعا.")
citation_text = (f"در hybrid دقت ارجاع {ltr(metric('hybrid', 'cite_precision')+'%')} و "
                 f"بازخوانی ارجاع {ltr(metric('hybrid', 'cite_recall')+'%')} است؛ در فاز اول این مقادیر "
                 f"به‌ترتیب {ltr(metric('phase1_dense', 'cite_precision')+'%')} و "
                 f"{ltr(metric('phase1_dense', 'cite_recall')+'%')} هستند. "
                 "تفاوت جفتی و فاصله‌های اطمینان در جدول مقایسه آمده‌اند. "
                 "دقت به تطابق ارجاعات ثبت‌شده با شناسه‌های طلایی و بازخوانی به پوشش شناسه‌های طلایی مربوط است؛ "
                 "این اعداد داوری معنایی تک‌تک ادعاها یا کافی‌بودن همهٔ شواهد نیستند.")
mask_detail = (f"در اجرای ماسک‌شده {ltr(ABST_SUMMARY['category_counts'].get('masked_abstained', 0))} امتناع، "
               f"{ltr(ABST_SUMMARY['category_counts'].get('masked_correct', 0))} پاسخ درست و "
               f"{ltr(ABST_SUMMARY['category_counts'].get('masked_wrong', 0))} پاسخ نادرست ثبت شد. "
               "امتناع نشان می‌دهد مدل در این تعداد طبق قرارداد از پاسخ خودداری کرده، نه اینکه در تمام موارد واقعاً شواهد ناکافی بوده است. "
               "برای ارزیابی بهتر، نمونه‌های پاسخ نادرست و اسناد باقی‌مانده باید دستی بررسی شوند.")
footer_text = ("پیش‌بینی‌های hidden فقط از نظر ساختاری بررسی می‌شوند و چون برچسب عمومی در دسترس نیست، "
               "عملکرد پاسخ آن‌ها ادعا نمی‌شود. تاریخ و commit بالا اطلاعات اعلام‌شده‌اند، نه تأیید مستقل نسخهٔ فاز دوم.")
styles = '''
@font-face{font-family:Vazir;src:url(data:font/ttf;base64,FONT)}
@page{size:A4;margin:14mm 14mm 16mm;@bottom-center{content:"صفحه " counter(page);font-family:Vazir;font-size:9pt;color:#6d7d91}}
body{font-family:Vazir,sans-serif;direction:rtl;color:#1e3048;font-size:10.5pt;line-height:1.8}
h1,h2,h3{color:#173e6c;line-height:1.5;break-after:avoid}h1{font-size:19pt;margin:2px 0}h2{font-size:14pt;border-right:4px solid #2785bb;background:#edf5fb;padding:5px 10px;margin:17px 0 9px}h3{font-size:11pt;margin:10px 0 5px}p{margin:7px 0 10px;text-align:right;orphans:3;widows:3}.bism{text-align:center;font-size:13pt;color:#244970;margin-bottom:12px}header{text-align:center;margin:0 0 12px}.eyebrow{color:#537695;font-size:9pt}.subtitle{color:#56718f;text-align:center;margin:3px 0}.identity{display:flex;background:#123e69;color:white;padding:11px 18px;margin:12px 0;border-radius:7px;gap:32px}.identity div{flex:1}.identity span{display:block;font-size:8.5pt;color:#d6e9fc}.identity strong{font-size:12pt;color:white}.latin{direction:ltr;unicode-bidi:isolate;display:inline-block;font-family:Vazir,sans-serif}table{width:100%;border-collapse:collapse;margin:8px 0 12px;font-size:9pt;line-height:1.55}th{background:#c8dfef;color:#143a60;text-align:center}td{background:#f4f9fc;text-align:center}tr:nth-child(even) td{background:#e8f2f8}th,td{border:0.7px solid #bfd0dd;padding:5px 6px;vertical-align:middle}tr{break-inside:avoid}.meta td:first-child{text-align:right;width:33%;font-weight:bold}.meta td:last-child{text-align:right;overflow-wrap:anywhere}.metrics{font-size:7.7pt}.metrics th,.metrics td{padding:4px 2px}.note{background:#f0f6fc;border-right:3px solid #4c91c3;padding:8px 12px;margin:8px 0}.warning{background:#fff7e9;border-right:3px solid #daa241;padding:8px 12px}figure{text-align:center;margin:8px 0}figure img{max-width:95%;max-height:5.4cm;object-fit:contain}figcaption{font-size:8pt;color:#657b91}.pagebreak{break-before:page}.small{font-size:9pt;color:#536b82}.case{background:#f6fafc;border:1px solid #d3e2ed;padding:8px 12px;margin:7px 0;break-inside:avoid}.case bdi{overflow-wrap:anywhere}
'''.replace('FONT',font_b64)

def document(body, compact=False):
    extra = ('body{font-size:9pt;line-height:1.55}h1{font-size:16pt}h2{font-size:11pt;margin:9px 0 5px}p{margin:4px 0 6px}.bism{font-size:11pt;margin-bottom:5px}.identity{padding:7px 12px;margin:7px 0}.identity strong{font-size:10pt}.meta{font-size:8pt;margin:5px 0}.meta th,.meta td{padding:3px}.metrics{font-size:7.2pt}.metrics td,.metrics th{padding:3px 1px}table{font-size:8pt;margin:5px 0}th,td{padding:3px 4px}.note{padding:5px 8px;margin:5px 0}figure img{max-height:3.5cm}' if compact else '')
    return '<!doctype html><html lang="fa" dir="rtl"><head><meta charset="utf-8"><style>'+styles+extra+'</style></head><body>'+body+'</body></html>'

# The official handout limits the submitted report to three pages.
compact_body = (header + '<h2>۱. داده و روش</h2><p>'+methods+'</p><h2>۲. نتایج کلی dev</h2>' + overall_table +
    '<p class="small">همهٔ اعداد درصد هستند. phase1_dense خروجی ذخیره‌شدهٔ فاز اول است.</p>' +
    '<div class="pagebreak"></div><h2>۳. مقایسه و تحلیل</h2><p>'+comparison+'</p>'+pair_table+
    '<div class="note">'+result_text+'</div><h3>تفکیک نوع پرسش: hybrid منهای phase1_dense</h3>'+ptype_table+
    '<p class="small">'+_type_why+'</p>'+
    '<div class="pagebreak"></div><h2>۴. امتناع و تحلیل خطا</h2>'+mask_table+'<p>'+mask_text+'</p>'+error_table+
    '<p>'+error_text+'</p><p class="small">'+citation_text+' '+footer_text+'</p>')
compact_html = document(compact_body, compact=True)
report_html_path = OUTPUT_DIR / 'report.html'
report_html_path.write_text(compact_html, encoding='utf-8')
compact_doc = HTML(string=compact_html, base_url=str(OUTPUT_DIR)).render()
if len(compact_doc.pages) > 3:
    raise RuntimeError(f"Submission report has {len(compact_doc.pages)} pages; handout allows 3. Adjust compact layout.")
REPORT_PATH = OUTPUT_DIR / 'report.pdf'
compact_doc.write_pdf(str(REPORT_PATH))

# A separate detailed report is not subject to the submission page limit.
examples = []
for category in ('retrieval_failure', 'packing_evidence_gap', 'generation_error', 'citation_gap'):
    subset = ERR_DF[ERR_DF['error'] == category]
    if subset.empty:
        continue
    qid = str(subset.iloc[0]['qid'])
    item = QDEV_BY_QID[qid]
    pred = next(x for x in DEV_PREDS['hybrid'] if str(x['qid']) == qid)
    packed = CONTEXT_USED['hybrid'].get(qid, [])
    examples.append(f'<div class="case"><b>{esc(error_labels[category])}</b> · {ltr(qid)}<br>'
        f'پرسش: {ltr(item["question"])}<br>پاسخ طلایی: {ltr(LABELS_BY_QID[qid]["answer"])}<br>'
        f'سندهای طلایی: {ltr(", ".join(gold_ids_of(qid)))}<br>'
        f'اسناد بازیابی‌شده: {ltr(", ".join(pred["retrieved"]))}<br>'
        f'سندهای واقعاً بسته‌بندی‌شده: {ltr(", ".join(packed) or "—")}<br>'
        f'پاسخ استخراج‌شده: {ltr(pred["answer"] or "—")} · امتناع: {ltr(pred["abstained"])} · '
        f'ارجاعات: {ltr(", ".join(pred["cited"]) or "—")}<br>'
        f'بخشی از خروجی خام: {ltr(RAW_OUTPUTS["hybrid"].get(qid, "")[:400] or "—")}</div>')
variant_pair = PAIRED.get('hybrid_sentence__vs__hybrid', {})
variant_rows = [[ltr(m), ltr(pmetric('hybrid_sentence__vs__hybrid',m))] for m in ('hitall@10','em','f1')]
detailed_body = (header + '<h2>۱. صورت مسئله و روش اجرا</h2><p>'+methods+'</p>'+
    '<p>بردار سندها از dataset خوانده می‌شود؛ فقدان یا ناسازگاری فایل به خطا منجر می‌شود. '
    'BM25 پس از همسان‌سازی IDF با rank_bm25 آزمون می‌شود؛ برابری با IDF پیش‌فرض کتابخانه ادعا نمی‌شود. '
    'رتبه‌های dense و BM25 به جای جمع نمرات ناهم‌مقیاس، با RRF ترکیب می‌شوند.</p>'+
    '<p>در آزمایش جمله، نمرهٔ جمله‌های هر سند با بیشینه تجمیع و همچنان در سطح سند گزارش می‌شود. '
    'شناسهٔ ارجاع باید در زمینهٔ واقعاً بسته‌بندی‌شده باشد؛ این شرط پشتیبانی معنایی سند از پاسخ را به‌تنهایی تضمین نمی‌کند.</p>'+
    '<h2>۲. نتایج کلی و مقایسه با فاز اول</h2>'+overall_table+'<p>'+comparison+'</p>'+pair_table+
    '<div class="note">'+result_text+'</div>'+figure('overall','مقایسهٔ معیارهای اصلی dev')+
    '<h2>۳. تحلیل به تفکیک نوع سؤال و دانه‌بندی</h2><p>جدول زیر اختلاف جفتی hybrid منهای dense ذخیره‌شدهٔ فاز اول را برای هر نوع نشان می‌دهد. '
    'علامت مثبت به معنی مقدار بیشتر hybrid است؛ نتیجهٔ هر نوع را همراه تعداد و فاصلهٔ اطمینان تفسیر کنید.</p>'+ptype_table+
    '<p>'+_type_note+' '+_type_why+'</p><h3>تغییرات پوشش کامل اسناد طلایی</h3>'+transition_table+'<h3>آزمایش سطح جمله در برابر hybrid سطح سند</h3>'+table(['معیار','sentence − hybrid؛ بازه ۹۵٪'],variant_rows)+
    '<p>تغییر واحد بردارسازی dense و تجمیع بیشینه، هم انتخاب شواهد و هم نتیجهٔ پاسخ را تغییر می‌دهد؛ '
    'بازهٔ اطمینان برای قضاوت جهت اثر و فایل‌های پیش‌بینی برای بررسی سؤال‌های خاص لازم است.</p>'+figure('by_type','مقایسهٔ نتایج به تفکیک نوع پرسش')+
    '<h2>۴. استناد و آزمایش امتناع با حذف اسناد طلایی</h2><p>'+citation_text+'</p>'+mask_table+'<p>'+mask_text+' '+mask_detail+'</p>'+figure('abstention','دسته‌های مشاهده‌شده در آزمایش حذف اسناد طلایی')+
    '<h2>۵. تحلیل خطا و نمونه‌ها</h2>'+error_table+'<p>'+error_text+'</p>'+''.join(examples)+
    '<p>نمونه‌ها نخستین مورد هر دسته‌اند و نمونه‌گیری تصادفی نمایندهٔ کل خطاها نیستند. '
    'فایل error_analysis.txt و خروجی خام برای بررسی دستی در بسته موجود است.</p>'+
    '<h2>۶. محدودیت‌ها و بازتولیدپذیری</h2><p>'+footer_text+'</p>'+
    '<p>ثابت RRF مطابق فرمول صورت پروژه است، نه نتیجهٔ تنظیم ابرپارامتر روی این مجموعه. '
    'محدودیت context، تفاوت prompt خروجی تاریخی، نرمال‌سازی پاسخ و وابستگی به داده‌های dev در تفسیر نتایج اهمیت دارند. '
    'از دستیار هوش مصنوعی برای توسعه و بازبینی استفاده شده است؛ اعداد بالا از خروجی همین اجرا و scorer به دست آمده‌اند.</p>')
detailed_html = document(detailed_body)
DETAILED_HTML_PATH = OUTPUT_DIR / 'report_detailed.html'
DETAILED_PATH = OUTPUT_DIR / 'report_detailed.pdf'
DETAILED_HTML_PATH.write_text(detailed_html, encoding='utf-8')
detailed_doc = HTML(string=detailed_html, base_url=str(OUTPUT_DIR)).render()
detailed_doc.write_pdf(str(DETAILED_PATH))
print(f"Submission report: {REPORT_PATH} ({len(compact_doc.pages)} pages)")
print(f"Detailed report: {DETAILED_PATH} ({len(detailed_doc.pages)} pages)")

SUBMISSION_ZIP = Path('/kaggle/working') / f'P2_{STD_ID}.zip'
if not SUBMISSION_ZIP.parent.exists():
    SUBMISSION_ZIP = OUTPUT_DIR / f'P2_{STD_ID}.zip'
with zipfile.ZipFile(SUBMISSION_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in (REPORT_PATH, report_html_path, DETAILED_PATH, DETAILED_HTML_PATH):
        zf.write(fp, arcname=fp.name)
    zf.write(PROMPT_FILE, arcname='prompts/phase2_strict.txt')
    if ERROR_TXT.exists():
        zf.write(ERROR_TXT, arcname='analysis/error_analysis.txt')
    artifact_names = {
        'abstention_analysis.csv', 'abstention_summary.json', 'bm25_reference_check.json',
        'dense_ranking_audit.json', 'error_taxonomy.csv', 'paired_by_type.json',
        'rankings_dev.json', 'rankings_hidden.json', 'summary_dev.json',
        'type_retrieval_transitions.json',
    }
    artifact_names.update(f'score_{s}.txt' for s in SCORES)
    artifact_names.update(f'paired_{k.replace("__vs__", "_vs_")}.txt' for k in PAIRED)
    for split, active in (('dev', DEV_SYSTEMS), ('hidden', HIDDEN_SYSTEMS)):
        for s in active:
            artifact_names.update((f'context_used_{split}_{s}.json', f'raw_outputs_{split}_{s}.json'))
    for name in sorted(artifact_names):
        path = ARTIFACT_DIR / name
        if path.is_file():
            zf.write(path, arcname=f'analysis/{path.name}')
    for path in sorted(FIG_DIR.glob('*.png')):
        zf.write(path, arcname=f'figures/{path.name}')
    for split, names in (('dev', list(DEV_SYSTEMS) + ['phase1_dense']), ('hidden', HIDDEN_SYSTEMS)):
        for name in names:
            run_dir = RUNS_DIR / split / name
            for filename in ('predictions.jsonl','run_config.yaml'):
                fp = run_dir / filename
                if not fp.is_file():
                    raise FileNotFoundError(f'Missing current-run output: {fp}')
                zf.write(fp, arcname=str(fp.relative_to(OUTPUT_DIR)))
with zipfile.ZipFile(SUBMISSION_ZIP) as zf:
    names = zf.namelist()
    assert 'report.pdf' in names and not any(n.endswith('.npz') for n in names)
    assert sum(n.endswith('predictions.jsonl') for n in names) == sum(n.endswith('run_config.yaml') for n in names)
print('submission:', SUBMISSION_ZIP)


Submission report: /kaggle/working/NLP_phase2/report.pdf (3 pages)
Detailed report: /kaggle/working/NLP_phase2/report_detailed.pdf (6 pages)
submission: /kaggle/working/P2_404722145.zip


## Submission checklist

- Attach the required document NPZ and matching metadata in the input dataset. The archive excludes embedding files.
- Fill in the real `GITHUB_COMMIT` in CONFIG, run all cells, and inspect `report.pdf`.
- Use `dev_limit=None` for submission; check hidden validation and the archive.
- `P2_<StudentID>.zip` contains the report, run files, figures and analysis artifacts. `report.pdf` is the official submission report and is checked against the three-page limit; `report_detailed.pdf` is a separate extended appendix.
